# Pollen in Sweden: analysis

Step 7 (Phase 5). Answers the 11 project questions using the clean `marts` tables built by dbt.

**Data:** Open-Meteo Air Quality API (CAMS model) + Open-Meteo Historical Weather, 6 cities, hourly, 2021 → yesterday.
**Note:** pollen values are model estimates, not measurements. This is pattern information, not medical advice. For treatment, consult your doctor.

| Section | # | Question | Who uses it |
|---|---|---|---|
| **EDA** | – | Pollen calendar, distribution, correlations, weather context | Everyone (overview) |
| **People with allergies** | 1 | When does each pollen season start, peak and end, per city? | People with allergies |
| | 2 | How many weeks earlier does birch start in the south than in the north? | People with allergies, travellers |
| | 3 | At what time of day is pollen lowest? | People with allergies |
| | 4 | How do temperature, rain and wind affect pollen? | People with allergies |
| | 5 | How many days does pollen take to rise again after rain? | People with allergies |
| | 6 | Which days combine high pollen and bad air quality? | People with allergies and asthma |
| | 7 | Which year had the most intense season, and why? | People with allergies, health services |
| | – | **Allergy risk calendar:** which weeks are safest for outdoor activities? | People with allergies |
| **Business questions** | 8 | In which weeks should pharmacies stock up on antihistamines, per region? | Pharmacies |
| | 9 | Which destinations are best in spring for people with allergies? | Tourism (travel agencies, hotels) |
| | 10 | Which dates have the lowest risk for outdoor events? | Event organisers (festivals, races) |
| | 11 | When to sell air purifiers and filters? | Home & retail |
| | 12 | A/B test design: does a campaign that follows the pollen sell more than one that follows the calendar? | Retail (marketing) |
| **Validation** | – | Do the model's Stockholm seasons match real measurements (CAMM)? | Everyone (trust in the data) |
| **Recommendations** | – | Concrete actions for people with allergies and for each business sector | Everyone |

In [33]:
# Setup: run this first after every kernel restart
from pathlib import Path

import altair as alt
import duckdb
import pandas as pd

# Project root = first folder upwards that contains pyproject.toml (works from notebooks/ or root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DB_PATH = ROOT / "data" / "polen.duckdb"


def query(sql):
    """Run SQL on the database and return a DataFrame.
    Opens and closes the connection each time, so dbt / ingest.py are never blocked."""
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        return con.sql(sql).df()


# Fixed orders used in every chart
CITIES = ["Malmö", "Göteborg", "Stockholm", "Uppsala", "Umeå", "Luleå"]  # south -> north
POLLEN = ["Alder", "Birch", "Grass", "Mugwort"]                           # season order
REGION = {"Malmö": "South", "Göteborg": "South", "Stockholm": "Middle",
          "Uppsala": "Middle", "Umeå": "North", "Luleå": "North"}
REGIONS = ["South", "Middle", "North"]

# Colours: one fixed colour per city / region / pollen type (same in every chart)
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
CITY_SCALE = alt.Scale(domain=CITIES, range=CATEGORICAL)
REGION_SCALE = alt.Scale(domain=REGIONS, range=CATEGORICAL[:3])
POLLEN_SCALE = alt.Scale(domain=POLLEN, range=CATEGORICAL[:4])
POLLEN3_SCALE = alt.Scale(domain=POLLEN[1:], range=CATEGORICAL[1:4])  # same colours, without alder

# Sequential (light = low, dark = high) and diverging (blue = below, grey = typical, red = above)
BLUE_RAMP = ["#f3f7fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = ["#256abf", "#9ec5f4", "#f0efec", "#f1a9a8", "#e34948"]
GRID = "#e1e0d9"

# Labels: values >= 1000 shown as K
K_LABELS = "datum.value >= 1000 ? format(datum.value / 1000, '~r') + 'K' : format(datum.value, '~r')"


def title(text, subtitle=None):
    """Bold chart title with optional subtitle."""
    return alt.Title(text, fontWeight="bold", anchor="start", subtitle=subtitle or "")


def season_days(pollen_types=("alder", "birch", "grass", "mugwort")):
    """Daily data inside each pollen season (start to end), one row per city, day and pollen type.
    ratio = pollen that day / AVERAGE pollen of that city's season and year -> 1 = average season day.
    (The average is used instead of the median because alder has many near-zero days, so its median is ~0.)"""
    types = ", ".join(f"'{t}'" for t in pollen_types)
    df = query(f"""
        SELECT d.*, s.pollen_type, s.year,
               CASE s.pollen_type WHEN 'alder' THEN d.alder_pollen_avg WHEN 'birch' THEN d.birch_pollen_avg
                                  WHEN 'grass' THEN d.grass_pollen_avg ELSE d.mugwort_pollen_avg END AS grains
        FROM marts.daily d
        JOIN marts.seasons s
          ON s.city = d.city AND d.date_local BETWEEN s.season_start AND s.season_end
        WHERE s.pollen_type IN ({types})
    """)
    df["ratio"] = df["grains"] / df.groupby(["city", "pollen_type", "year"])["grains"].transform("mean")
    df["pollen_type"] = df["pollen_type"].str.capitalize()
    return df


## EDA: pollen calendar

Average pollen per week of the year (2021–2026), per city and pollen type.

In [34]:
weekly = query("SELECT * FROM marts.weekly WHERE iso_week <= 52")

calendar = (
    weekly.melt(id_vars=["city", "iso_week"],
                value_vars=[f"{p.lower()}_pollen_avg" for p in POLLEN],
                var_name="pollen", value_name="grains")
    .assign(pollen=lambda d: d["pollen"].str.replace("_pollen_avg", "").str.capitalize())
    .groupby(["pollen", "city", "iso_week"], as_index=False)["grains"].mean()
)

panels = [
    alt.Chart(calendar.query("pollen == @p"))
    .mark_rect(stroke="white", strokeWidth=1)
    .encode(
        x=alt.X("iso_week:O", title="Week of the year",
                axis=alt.Axis(values=list(range(5, 53, 5)), labelAngle=0)),
        y=alt.Y("city:N", sort=CITIES, title=None),
        color=alt.Color("grains:Q", title="grains/m³",
                        scale=alt.Scale(range=BLUE_RAMP, nice=True),
                        legend=alt.Legend(labelExpr=K_LABELS, gradientLength=110)),
        tooltip=["city:N", alt.Tooltip("iso_week:O", title="week"),
                 alt.Tooltip("grains:Q", format=".1f", title="grains/m³")],
    )
    .properties(width=620, height=130, title=alt.Title(p, fontWeight="bold", anchor="start"))
    for p in POLLEN
]

(
    alt.vconcat(*panels)
    .resolve_scale(color="independent")
    .properties(title=alt.Title("Pollen calendar: average pollen per week, 2021–2026", fontWeight="bold",
                                subtitle="Cities south (top) to north (bottom). Each panel has its own colour scale"))
    .configure_view(stroke=None)
)

alt.VConcatChart(...)

**What the calendar shows**
- **Four seasons in a row:** alder (Mar–Apr) → birch (mid-Apr–May) → grass (Jun–Jul) → mugwort (mid-Jul–Aug).
- **Birch is by far the strongest:** weekly averages up to ~330 grains/m³, vs ~25 for grass and ~10 for mugwort.
- **Birch moves south → north:** the dark cells shift about 4 weeks later in Umeå and Luleå.
- **Grass is stronger in the south and middle** (Malmö, Göteborg, Uppsala) and weaker in the north.
- **Model limitations:** alder is concentrated in Göteborg, and mugwort happens in the same weeks in every city. Both look like simplifications of the CAMS model rather than real differences → check against measurements (CAMM, Pollenrapporten).

## EDA: distribution of daily pollen per city

How high is pollen on a normal season day, and how extreme are the bad days? Only days inside each season. Log scale: each step = 10× more pollen.

In [35]:
in_season = season_days(("birch", "grass"))

(
    alt.Chart(in_season.query("grains >= 1"))
    .mark_boxplot(size=22, color="#3987e5", median={"color": "#0d366b"}, outliers={"size": 6, "opacity": 0.3})
    .encode(
        x=alt.X("city:N", sort=CITIES, title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("grains:Q", scale=alt.Scale(type="log"), title="Daily pollen (grains/m³, log scale)",
                axis=alt.Axis(values=[1, 3, 10, 30, 100, 300, 1000, 3000], labelExpr=K_LABELS, gridColor=GRID)),
    )
    .properties(width=300, height=260)
    .facet(column=alt.Column("pollen_type:N", title=None, header=alt.Header(labelFontWeight="bold", labelFontSize=12)))
    .resolve_scale(y="independent")
    .properties(title=title("Daily pollen on season days, per city",
                            "Days with ≥ 1 grain/m³. Box = middle 50% of days, line = median, dots = extreme days. 2021–2026"))
    .configure_view(stroke=None)
)

alt.FacetChart(...)

- **Birch:** a typical season day has ~50–150 grains/m³ (median per city), and the worst days reach ~1K. Göteborg, Uppsala and Umeå have the highest typical levels.
- **Grass:** much lower levels (median ~4–12) and a narrower range. Highest in the south and middle (Malmö, Göteborg, Uppsala), lowest in the north.

## EDA: how the daily variables relate to each other

Spearman correlation between daily values on birch-season days (−1 = opposite, 0 = no relation, +1 = move together).

In [36]:
corr_cols = {
    "birch_pollen_avg": "Birch pollen", "pm2_5_avg": "PM2.5 (fine particles)", "ozone_avg": "Ozone", "no2_avg": "NO2 (nitrogen dioxide)",
    "temperature_max_c": "Max temperature", "humidity_avg_pct": "Humidity",
    "precipitation_mm": "Rain", "wind_speed_avg_kmh": "Wind speed",
}
birch_days = season_days(("birch",))
corr = (birch_days[list(corr_cols)].rename(columns=corr_cols).corr(method="spearman")
        .reset_index().melt(id_vars="index", var_name="var2", value_name="rho")
        .rename(columns={"index": "var1"}))
order = list(corr_cols.values())
# keep each pair once (lower triangle), drop the diagonal (always 1)
corr = corr[corr["var1"].map(order.index) > corr["var2"].map(order.index)]

base = alt.Chart(corr).encode(
    x=alt.X("var2:N", sort=order, title=None, axis=alt.Axis(labelAngle=-40)),
    y=alt.Y("var1:N", sort=order, title=None),
)
(
    alt.layer(
        base.mark_rect(stroke="white", strokeWidth=2).encode(
            color=alt.Color("rho:Q", title="Correlation", scale=alt.Scale(domain=[-1, 1], range=DIVERGING, interpolate="rgb")),
            tooltip=["var1:N", "var2:N", alt.Tooltip("rho:Q", format=".2f")],
        ),
        base.mark_text(fontSize=11).encode(
            text=alt.Text("rho:Q", format=".2f"),
            color=alt.condition("abs(datum.rho) > 0.6", alt.value("white"), alt.value("#0b0b0b")),
        ),
    )
    .properties(width=400, height=380, title=title("Correlation between daily variables (birch season)",
                                                   "Spearman rank correlation, all cities 2021–2026"))
    .configure_view(stroke=None)
)

alt.LayerChart(...)

- **Birch pollen goes up with temperature (+0.45)** and down with **rain (−0.30)** and **humidity (−0.27)**. Wind has a small negative effect (−0.12).
- **PM2.5 (fine particles smaller than 2.5 µm) and NO2 (nitrogen dioxide) move together (+0.66):** both come mostly from traffic and combustion.
- **Ozone rises with wind and dry air**, not with pollen → pollen and ozone peaks are mostly independent.
- Correlation ≠ cause: warm days are also later in the season, when birch is flowering. Q4 separates these effects.

## EDA: weather context per city

Average temperature and rain per month (2021–2026). Two separate charts (not two y-axes on one chart).

In [37]:
monthly = query("""
    SELECT city, month(date_local) AS month,
           avg(temperature_avg_c)  AS temperature_c,
           sum(precipitation_mm) / count(DISTINCT year(date_local)) AS rain_mm
    FROM marts.daily
    GROUP BY ALL
""")

def monthly_line(field, y_title, chart_title):
    return (
        alt.Chart(monthly)
        .mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=30))
        .encode(
            x=alt.X("month:O", title="Month", axis=alt.Axis(labelAngle=0)),
            y=alt.Y(f"{field}:Q", title=y_title, axis=alt.Axis(gridColor=GRID)),
            color=alt.Color("city:N", sort=CITIES, scale=CITY_SCALE, title="City"),
            tooltip=["city:N", "month:O", alt.Tooltip(f"{field}:Q", format=".1f")],
        )
        .properties(width=320, height=220, title=title(chart_title))
    )

(
    alt.hconcat(monthly_line("temperature_c", "°C", "Average temperature"),
                monthly_line("rain_mm", "mm per month", "Average rain"))
    .configure_view(stroke=None)
)

alt.HConcatChart(...)

- The north is **~4–7 °C colder in spring** (March–May): that is why birch flowers ~3–4 weeks later in Umeå and Luleå.
- Rain is fairly similar between cities; **summer (Jul–Aug) is the wettest period**, which coincides with the grass and mugwort seasons.

## People with allergies (Q1–Q7)

Questions for people with pollen allergy who want to plan outdoor activities: when and where pollen is high, at what time of day, and how weather and air quality change it. The section ends with an **allergy risk calendar** per week.

## Q1. When does each pollen season start, peak and end, per city?

Season = from the day 5% of the year's pollen has fallen (start) to 95% (end); peak = day with the highest daily average (`marts.seasons`). Typical season = median over 2021–2026.

In [38]:
seasons = query("SELECT * FROM marts.seasons")

# Put every year on the same calendar (year 2000) so seasons can be compared and averaged
for col in ["season_start", "peak_date", "season_end"]:
    seasons[col + "_md"] = pd.to_datetime(seasons[col].astype(str).str.replace(r"^\d{4}", "2000", regex=True))

typical = (
    seasons.groupby(["pollen_type", "city"], as_index=False)
    .agg(start=("season_start_md", "median"), peak=("peak_date_md", "median"), end=("season_end_md", "median"),
         start_week=("start_week", "median"), peak_week=("peak_week", "median"), end_week=("end_week", "median"))
)
typical["pollen_type"] = typical["pollen_type"].str.capitalize()

# Table: typical dates per pollen type and city (south -> north)
(typical.assign(city=pd.Categorical(typical["city"], CITIES, ordered=True),
                pollen_type=pd.Categorical(typical["pollen_type"], POLLEN, ordered=True))
        .sort_values(["pollen_type", "city"])
        .assign(start=lambda d: d["start"].dt.strftime("%d %b"),
                peak=lambda d: d["peak"].dt.strftime("%d %b"),
                end=lambda d: d["end"].dt.strftime("%d %b"))
        [["pollen_type", "city", "start", "peak", "end", "start_week", "peak_week", "end_week"]]
        .reset_index(drop=True))

,pollen_type,city,start,peak,end,start_week,peak_week,end_week
0,Alder,Malmö,20 Feb,26 Mar,14 Apr,7.5,12.5,15.0
1,Alder,Göteborg,27 Feb,02 Apr,15 Apr,8.5,13.5,15.5
2,Alder,Stockholm,08 Mar,04 Apr,22 Apr,10.5,14.0,16.5
3,Alder,Uppsala,04 Mar,08 Apr,27 Apr,9.5,14.5,17.5
4,Alder,Umeå,25 Mar,07 Apr,15 May,12.5,14.5,19.5
5,Alder,Luleå,16 Mar,17 Apr,21 May,11.0,16.0,21.0
6,Birch,Malmö,14 Apr,29 Apr,15 May,15.5,18.0,20.0
7,Birch,Göteborg,15 Apr,26 Apr,11 May,16.0,17.0,19.0
8,Birch,Stockholm,15 Apr,08 May,21 May,15.5,18.5,21.0
9,Birch,Uppsala,18 Apr,06 May,20 May,16.0,18.5,20.5


In [39]:
base = alt.Chart(typical).encode(y=alt.Y("city:N", sort=CITIES, title=None))

bars = base.mark_bar(height=12, cornerRadius=4, color="#86b6ef").encode(
    x=alt.X("start:T", title=None,
            scale=alt.Scale(domain=["2000-02-01", "2000-09-01"]),
            axis=alt.Axis(format="%b", tickCount="month", gridColor="#e1e0d9")),
    x2="end:T",
    tooltip=["city:N", alt.Tooltip("start:T", format="%d %b"),
             alt.Tooltip("peak:T", format="%d %b"), alt.Tooltip("end:T", format="%d %b")],
)
peaks = base.mark_point(filled=True, size=70, color="#184f95", stroke="white", strokeWidth=2).encode(
    x="peak:T",
    tooltip=["city:N", alt.Tooltip("peak:T", format="%d %b", title="peak")],
)

(
    alt.layer(bars, peaks)
    .properties(width=620, height=150)
    .facet(row=alt.Row("pollen_type:N", sort=POLLEN, title=None,
                       header=alt.Header(labelFontWeight="bold", labelAngle=0, labelAlign="left", labelFontSize=12)))
    .properties(title=alt.Title("When does each pollen season happen?", fontWeight="bold",
                                subtitle="Typical season (median 2021–2026): bar = start (5%) to end (95%), dot = peak day"))
    .configure_view(stroke=None)
)

alt.FacetChart(...)

**Answer Q1** (typical dates, median 2021–2026; start → **peak** → end)

| Pollen | South (Malmö) | Middle (Stockholm) | North (Luleå) |
|---|---|---|---|
| **Alder** | 20 Feb → **26 Mar** → 14 Apr | 8 Mar → **4 Apr** → 22 Apr | 16 Mar → **17 Apr** → 21 May |
| **Birch** | 14 Apr → **29 Apr** → 15 May | 15 Apr → **8 May** → 21 May | 12 May → **21 May** → 6 Jun |
| **Grass** | 1 Jun → **26 Jun** → 6 Aug | 6 Jun → **29 Jun** → 9 Aug | 14 Jun → **30 Jun** → 30 Jul |
| **Mugwort** | 9 Jul → **25 Jul** → 17 Aug | 13 Jul → **26 Jul** → 16 Aug | 12 Jul → **23 Jul** → 15 Aug |

- The pollen year runs from **late February (alder, south) to mid-August (mugwort)**.
- **Birch**, the main allergy trigger, is short and intense: ~3–5 weeks. It peaks in **late April/early May** in the south and middle, and around **20 May** in the north.
- **Grass** has the longest season (~2 months, June–early August) everywhere.
- Season dates vary from year to year: birch start moves up to ±3 weeks (std ≈ 2 weeks), so the **typical** dates are a guide, not a fixed calendar.

## Q2. How many weeks earlier does each pollen season start in the south than in the north?

For **each pollen type and city**: how many weeks **after Malmö** (the southernmost city) the season starts.
Dot = median 2021–2026, line = range between the earliest and latest year.

In [42]:
starts = query("SELECT city, pollen_type, year, season_start FROM marts.seasons")
starts["pollen_type"] = starts["pollen_type"].str.capitalize()
starts["season_start"] = pd.to_datetime(starts["season_start"])

# Gap to Malmö in the same year and pollen type (positive = later than Malmö)
malmo = (starts.query("city == 'Malmö'")[["pollen_type", "year", "season_start"]]
         .rename(columns={"season_start": "malmo_start"}))
gaps = starts.merge(malmo, on=["pollen_type", "year"])
gaps["gap_weeks"] = (gaps["season_start"] - gaps["malmo_start"]).dt.days / 7

gap_summary = (gaps.groupby(["pollen_type", "city"], as_index=False)["gap_weeks"]
               .agg(median="median", earliest="min", latest="max"))

# Table: median gap in weeks vs Malmö, per city (rows) and pollen type (columns)
(gap_summary.pivot(index="city", columns="pollen_type", values="median")
            .reindex(index=CITIES, columns=POLLEN).round(1))

pollen_type,Alder,Birch,Grass,Mugwort
city,,,,
Malmö,0.0,0.0,0.0,0.0
Göteborg,0.2,-0.1,0.5,0.2
Stockholm,1.7,0.1,1.0,0.4
Uppsala,1.4,0.4,1.1,0.4
Umeå,3.6,2.9,2.6,0.0
Luleå,2.5,3.6,2.1,0.1


In [43]:
def gap_panel(pollen):
    data = gap_summary.query("pollen_type == @pollen")
    y = alt.Y("city:N", sort=CITIES, title=None)
    x_scale = alt.Scale(domain=[-1.5, 9])
    zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="#898781", strokeDash=[4, 4]).encode(x="x:Q")
    ranges = alt.Chart(data).mark_rule(strokeWidth=2, color="#c3c2b7").encode(
        y=y, x=alt.X("earliest:Q", scale=x_scale), x2="latest:Q")
    dots = alt.Chart(data).mark_point(filled=True, size=110, opacity=1, stroke="white", strokeWidth=2).encode(
        y=y,
        x=alt.X("median:Q", scale=x_scale, title="Weeks after Malmö", axis=alt.Axis(gridColor=GRID)),
        color=alt.Color("pollen_type:N", scale=POLLEN_SCALE, legend=None),
        tooltip=["city:N", "pollen_type:N", alt.Tooltip("median:Q", format=".1f", title="median weeks"),
                 alt.Tooltip("earliest:Q", format=".1f"), alt.Tooltip("latest:Q", format=".1f")],
    )
    return alt.layer(zero, ranges, dots).properties(width=260, height=170, title=title(pollen))

(
    alt.vconcat(alt.hconcat(gap_panel("Alder"), gap_panel("Birch")),
                alt.hconcat(gap_panel("Grass"), gap_panel("Mugwort")))
    .properties(title=title("How much later does each season start than in Malmö?",
                            "Dot = median 2021–2026, grey line = earliest to latest year, dashed line = same week as Malmö"))
    .configure_view(stroke=None)
)

alt.VConcatChart(...)

### Detail: birch per year, by region

Birch is the main allergy trigger, so here is its south → north gap year by year (South = Malmö + Göteborg, Middle = Stockholm + Uppsala, North = Umeå + Luleå).

In [44]:
birch = query("SELECT city, year, season_start FROM marts.seasons WHERE pollen_type = 'birch'")
birch["region"] = birch["city"].map(REGION)
birch["season_start"] = pd.to_datetime(birch["season_start"])

starts = birch.groupby(["year", "region"], as_index=False)["season_start"].mean()
starts["start_md"] = pd.to_datetime(starts["season_start"].dt.strftime("2000-%m-%d"))

gap = starts.pivot(index="year", columns="region", values="season_start")
gap["gap_days"] = (gap["North"] - gap["South"]).dt.days
gap["gap_weeks"] = (gap["gap_days"] / 7).round(1)
print(f"South → North gap: median {gap['gap_days'].median():.0f} days "
      f"(≈ {gap['gap_days'].median() / 7:.1f} weeks), range {gap['gap_days'].min()}–{gap['gap_days'].max()} days")
gap[["gap_days", "gap_weeks"]]

South → North gap: median 23 days (≈ 3.3 weeks), range 12–42 days


region,gap_days,gap_weeks
year,,
2021,23,3.3
2022,23,3.3
2023,24,3.4
2024,42,6.0
2025,12,1.7
2026,22,3.1


In [45]:
lines = starts.pivot(index="year", columns="region", values="start_md").reset_index()
lines["label"] = gap["gap_weeks"].values
lines["label"] = lines["label"].map(lambda w: f"{w:.1f} weeks")

y = alt.Y("year:O", title=None, sort="descending")
dumbbell = alt.Chart(lines).mark_rule(strokeWidth=2, color="#c3c2b7").encode(y=y, x="South:T", x2="North:T")
dots = alt.Chart(starts).mark_point(filled=True, size=140, opacity=1, stroke="white", strokeWidth=2).encode(
    y=y,
    x=alt.X("start_md:T", title="Birch season start", scale=alt.Scale(domain=["2000-03-25", "2000-05-25"]),
            axis=alt.Axis(format="%d %b", gridColor=GRID)),
    color=alt.Color("region:N", sort=REGIONS, scale=REGION_SCALE, title="Region"),
    tooltip=["region:N", "year:O", alt.Tooltip("season_start:T", format="%d %b %Y")],
)
labels = alt.Chart(lines).mark_text(align="left", dx=10, fontSize=11, color="#52514e").encode(
    y=y, x="North:T", text="label:N")

(
    alt.layer(dumbbell, dots, labels)
    .properties(width=560, height=260, title=title("Birch starts ~3 weeks earlier in the south",
                                                   "Average season start per region; label = South → North gap"))
    .configure_view(stroke=None)
)

alt.LayerChart(...)

**Answer Q2** (median gap vs Malmö, in weeks):

| City | Alder | Birch | Grass | Mugwort |
|---|---|---|---|---|
| Göteborg | 0.2 | −0.1 | 0.5 | 0.2 |
| Stockholm | 1.7 | 0.1 | 1.0 | 0.4 |
| Uppsala | 1.4 | 0.4 | 1.1 | 0.4 |
| Umeå | **3.6** | **2.9** | **2.6** | 0.0 |
| Luleå | 2.5 | **3.6** | 2.1 | 0.1 |

- **The gap is largest in spring and shrinks in summer:** alder and birch start **~3–3.5 weeks** later in the north, grass **~2–2.5 weeks**, and mugwort starts in the **same week everywhere** (a likely CAMS model simplification).
- **South and middle start almost together** for birch (Stockholm/Uppsala within ~3 days of Malmö); for alder and grass the middle is ~1–1.5 weeks later. The big jump is always between the middle and the **north**.
- **Göteborg starts with Malmö** for every type.
- **Predictability differs by type:** grass and mugwort start within ~1 week of their typical date every year (short grey lines), while alder and birch can shift by several weeks between years (long grey lines) → spring seasons need a bigger safety margin.
- **Birch per year:** the south → north gap varies from **12 days (2025, warm spring)** to **42 days (2024)**; median ~23 days (≈ 3.3 weeks). The grey lines in the chart show the same: in some years the north starts almost with the south, in others 6–7 weeks later.

## Q3. At what time of day is pollen lowest?

Average pollen per hour of the day (Swedish time) on season days, as an index: **100 = the season's daily average**.

In [10]:
hourly_profile = query("""
    WITH h AS (
        SELECT h.*, s.pollen_type
        FROM marts.hourly h
        JOIN marts.seasons s
          ON s.city = h.city AND s.year = year(h.date_local)
         AND h.date_local BETWEEN s.season_start AND s.season_end
    )
    SELECT pollen_type, hour_local,
           avg(CASE pollen_type WHEN 'alder' THEN alder_pollen WHEN 'birch' THEN birch_pollen
                                WHEN 'grass' THEN grass_pollen ELSE mugwort_pollen END) AS grains
    FROM h
    GROUP BY ALL
""")
hourly_profile["pollen_type"] = hourly_profile["pollen_type"].str.capitalize()
hourly_profile["index"] = (hourly_profile["grains"]
                           / hourly_profile.groupby("pollen_type")["grains"].transform("mean") * 100)

lowest = hourly_profile.loc[hourly_profile.groupby("pollen_type")["index"].idxmin()]
lowest[["pollen_type", "hour_local", "index"]].round(0)

,pollen_type,hour_local,index
41,Alder,8,20.0
84,Birch,5,52.0
44,Grass,6,62.0
75,Mugwort,5,74.0


In [11]:
band = alt.Chart(pd.DataFrame({"start": [4], "end": [8]})).mark_rect(color="#f0efec").encode(x="start:Q", x2="end:Q")
band_label = alt.Chart(pd.DataFrame({"x": [6], "y": [225], "t": ["lowest: 04–08"]})).mark_text(
    fontSize=11, color="#52514e").encode(x="x:Q", y="y:Q", text="t:N")
ref = alt.Chart(pd.DataFrame({"y": [100]})).mark_rule(strokeDash=[4, 4], color="#898781").encode(y="y:Q")
lines = alt.Chart(hourly_profile).mark_line(strokeWidth=2).encode(
    x=alt.X("hour_local:Q", title="Hour of the day (Swedish time)", scale=alt.Scale(domain=[0, 23]),
            axis=alt.Axis(values=list(range(0, 24, 2)), gridColor=GRID)),
    y=alt.Y("index:Q", title="Pollen index (100 = daily average)", axis=alt.Axis(gridColor=GRID)),
    color=alt.Color("pollen_type:N", sort=POLLEN, scale=POLLEN_SCALE, title="Pollen"),
    tooltip=["pollen_type:N", "hour_local:Q", alt.Tooltip("index:Q", format=".0f")],
)
end_labels = alt.Chart(hourly_profile.query("hour_local == 23")).mark_text(align="left", dx=6, fontSize=11).encode(
    x="hour_local:Q", y="index:Q", text="pollen_type:N", color=alt.value("#52514e"))

(
    alt.layer(band, band_label, ref, lines, end_labels)
    .properties(width=560, height=300, title=title("Pollen is lowest in the early morning",
                                                   "Average hourly pollen on season days, 2021–2026 (index, 100 = daily average)"))
    .configure_view(stroke=None)
)

alt.LayerChart(...)

### Per city

Same index per pollen type **and city** (100 = that city's daily average in the season). Blue = below average, red = above.

In [46]:
hourly_city = query("""
    WITH h AS (
        SELECT h.*, s.pollen_type
        FROM marts.hourly h
        JOIN marts.seasons s
          ON s.city = h.city AND s.year = year(h.date_local)
         AND h.date_local BETWEEN s.season_start AND s.season_end
    )
    SELECT city, pollen_type, hour_local,
           avg(CASE pollen_type WHEN 'alder' THEN alder_pollen WHEN 'birch' THEN birch_pollen
                                WHEN 'grass' THEN grass_pollen ELSE mugwort_pollen END) AS grains
    FROM h
    GROUP BY ALL
""")
hourly_city["pollen_type"] = hourly_city["pollen_type"].str.capitalize()
hourly_city["index"] = (hourly_city["grains"]
                        / hourly_city.groupby(["city", "pollen_type"])["grains"].transform("mean") * 100)

# Lowest and highest hour per pollen type and city
low = hourly_city.loc[hourly_city.groupby(["pollen_type", "city"])["index"].idxmin()]
high = hourly_city.loc[hourly_city.groupby(["pollen_type", "city"])["index"].idxmax()]
best_hours = low.merge(high, on=["pollen_type", "city"], suffixes=("_low", "_high"))
(best_hours.pivot(index="city", columns="pollen_type", values="hour_local_low")
           .reindex(index=CITIES, columns=POLLEN).rename_axis(columns="lowest hour"))

lowest hour,Alder,Birch,Grass,Mugwort
city,,,,
Malmö,8,7,6,6
Göteborg,8,6,6,6
Stockholm,13,6,6,5
Uppsala,6,6,6,5
Umeå,8,5,5,5
Luleå,7,5,5,6


In [47]:
def hour_panel(pollen):
    return (
        alt.Chart(hourly_city.query("pollen_type == @pollen"))
        .mark_rect(stroke="white", strokeWidth=1)
        .encode(
            x=alt.X("hour_local:O", title="Hour of the day (Swedish time)", axis=alt.Axis(labelAngle=0)),
            y=alt.Y("city:N", sort=CITIES, title=None),
            color=alt.Color("index:Q", title="Index (100 = daily average)",
                            scale=alt.Scale(domain=[25, 100, 200], range=[DIVERGING[0], DIVERGING[2], DIVERGING[4]],
                                            interpolate="rgb", clamp=True)),
            tooltip=["city:N", "hour_local:O", alt.Tooltip("index:Q", format=".0f")],
        )
        .properties(width=560, height=140, title=title(pollen))
    )

(
    alt.vconcat(*[hour_panel(p) for p in POLLEN])
    .properties(title=title("Pollen by hour of the day, per pollen type and city",
                            "Index per city (100 = daily average in the season) · blue = below, red = above, 2021–2026"))
    .configure_view(stroke=None)
)

alt.VConcatChart(...)

**Answer Q3:** pollen is **lowest early in the morning (05:00–08:00)** for **all four types in all six cities**, and **highest in the afternoon/evening**.

| Pollen | Lowest hour (per city) | Highest hour (per city) | Notes |
|---|---|---|---|
| **Birch** | 05–07 | 18–20 (Umeå 14) | Strongest daily cycle in Göteborg (from ~0.3× to ~1.6× the average) |
| **Grass** | 05–06 | 19–21 | Clear cycle in the south and middle; weaker in the north (Luleå 0.85–1.13×) |
| **Mugwort** | 05–06 | 21–22 (Göteborg 11) | Late peak, in the evening |
| **Alder** | 06–08 (Stockholm 13) | 15–21 | Very low levels outside Göteborg → noisier pattern |

- **The daily cycle is stronger in the south and middle** than in the north: in Luleå pollen stays closer to the average all day.
- **Tip for allergy sufferers:** outdoor exercise or airing the home is best **early in the morning** everywhere; avoid late afternoon and evening on high-pollen days.
- Caveat: the hourly cycle comes from the CAMS model (emission + transport), so the shape is reliable but exact hours are approximate.

## Q4. How do temperature, rain and wind affect birch, grass and mugwort, per city?

To remove the "time of season" effect, each day is compared with **its own season's average day**: ratio = pollen that day / average of that city's season (1 = average day, 2 = double).
Birch, grass and mugwort, first all cities together, then **per city**.

**Alder is left out:** in the CAMS model alder is near zero outside Göteborg (most season days < 1 grain/m³), so its weather effects are noise rather than signal (weak data).

In [48]:
w = season_days(("birch", "grass", "mugwort"))   # alder left out: weak data
w["Rain"] = pd.cut(w["precipitation_mm"], [-1, 0.1, 2, 1000], labels=["Dry (<0.1 mm)", "Light (0.1–2 mm)", "Rainy (>2 mm)"])
w["Wind"] = pd.cut(w["wind_speed_avg_kmh"], [-1, 10, 15, 20, 1000], labels=["<10 km/h", "10–15", "15–20", ">20 km/h"])
w["Max temperature"] = pd.cut(w["temperature_max_c"], [-50, 10, 15, 20, 25, 50], labels=["<10 °C", "10–15", "15–20", "20–25", ">25 °C"])

effects = pd.concat([
    w.groupby(["pollen_type", var], observed=True)["ratio"].agg(["mean", "size"]).reset_index()
     .rename(columns={var: "group"}).assign(variable=var)
    for var in ["Rain", "Wind", "Max temperature"]
])
effects = effects[effects["size"] >= 20]  # hide groups with too few days
effects.pivot_table(index=["variable", "group"], columns="pollen_type", values="mean", observed=True).reindex(columns=POLLEN[1:]).round(2)

pollen_type                       Birch  Grass  Mugwort
variable        group                                  
Max temperature 10–15              1.75   0.69      NaN
                15–20              2.74   1.09     1.13
                20–25              3.61   1.31     1.34
                <10 °C             0.71    NaN      NaN
                >25 °C              NaN   1.79     1.89
Rain            Dry (<0.1 mm)      2.27   1.62     1.75
                Light (0.1–2 mm)   1.45   1.20     1.27
                Rainy (>2 mm)      1.11   0.86     0.89
Wind            10–15              2.02   1.27     1.26
                15–20              1.52   1.18     1.04
                <10 km/h           1.87   1.36     1.65
                >20 km/h           1.60   1.01     0.90

In [49]:
def effect_chart(var):
    data = effects.query("variable == @var")
    order = list(w[var].cat.categories)
    lines = alt.Chart(data).mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=60, filled=True)).encode(
        x=alt.X("group:N", sort=order, title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q", title="Pollen vs average season day (×)", scale=alt.Scale(domain=[0, 2]),
                axis=alt.Axis(gridColor=GRID)),
        color=alt.Color("pollen_type:N", sort=POLLEN, scale=POLLEN3_SCALE, title="Pollen"),
        tooltip=["pollen_type:N", "group:N", alt.Tooltip("mean:Q", format=".2f", title="× average"), "size:Q"],
    )
    ref = alt.Chart(pd.DataFrame({"y": [1]})).mark_rule(strokeDash=[4, 4], color="#898781").encode(y="y:Q")
    return alt.layer(ref, lines).properties(width=250, height=240, title=title(var))

(
    alt.hconcat(effect_chart("Rain"), effect_chart("Wind"), effect_chart("Max temperature"))
    .properties(title=title("Weather effect on pollen (within the season), all cities",
                            "Mean ratio vs the average day of the same season; dashed line = average (1×). Groups with < 20 days hidden"))
    .configure_view(stroke=None)
)

alt.HConcatChart(...)

### Per city

One number per city and pollen type: **how much pollen there is on one kind of day compared with the opposite kind**.
- **Rain:** rainy days (> 2 mm) vs dry days (< 0.1 mm). 0.5× = half the pollen when it rains.
- **Wind:** the windiest third of days vs the calmest third (per city and pollen type).
- **Temperature:** the warmest third of days vs the coolest third.

Below 1 (blue) = less pollen, above 1 (red) = more pollen.

In [50]:
def city_effects(g):
    dry, rainy = g[g["precipitation_mm"] < 0.1], g[g["precipitation_mm"] >= 2]
    calm_cut, windy_cut = g["wind_speed_avg_kmh"].quantile([1 / 3, 2 / 3])
    cool_cut, warm_cut = g["temperature_max_c"].quantile([1 / 3, 2 / 3])
    return pd.Series({
        "Rain (rainy vs dry)": rainy["ratio"].mean() / dry["ratio"].mean(),
        "Wind (windy vs calm)": g.loc[g["wind_speed_avg_kmh"] >= windy_cut, "ratio"].mean()
                                / g.loc[g["wind_speed_avg_kmh"] <= calm_cut, "ratio"].mean(),
        "Temperature (warm vs cool)": g.loc[g["temperature_max_c"] >= warm_cut, "ratio"].mean()
                                      / g.loc[g["temperature_max_c"] <= cool_cut, "ratio"].mean(),
    })

city_effect = (w.groupby(["pollen_type", "city"])[["precipitation_mm", "wind_speed_avg_kmh", "temperature_max_c", "ratio"]]
               .apply(city_effects).reset_index()
               .melt(id_vars=["pollen_type", "city"], var_name="effect", value_name="times"))

city_effect.pivot_table(index=["effect", "city"], columns="pollen_type", values="times").reindex(columns=POLLEN[1:]).round(2)

pollen_type                           Alder  Birch  Grass  Mugwort
effect                     city                                   
Rain (rainy vs dry)        Göteborg     NaN   0.52   0.50     0.53
                           Luleå        NaN   0.72   0.45     0.31
                           Malmö        NaN   0.62   0.55     0.67
                           Stockholm    NaN   0.48   0.56     0.63
                           Umeå         NaN   0.33   0.49     0.37
                           Uppsala      NaN   0.35   0.59     0.59
Temperature (warm vs cool) Göteborg     NaN   2.16   1.66     2.18
                           Luleå        NaN   4.31   1.65     1.30
                           Malmö        NaN   1.67   1.59     0.99
                           Stockholm    NaN   3.81   1.40     1.31
                           Umeå         NaN   4.52   1.66     2.02
                           Uppsala      NaN   5.35   1.57     1.31
Wind (windy vs calm)       Göteborg     NaN   0.79   0.66     0.51
                           Luleå        NaN   0.87   0.93     0.61
                           Malmö        NaN   1.27   0.76     1.03
                           Stockholm    NaN   0.97   0.94     0.62
                           Umeå         NaN   1.10   0.86     0.51
                           Uppsala      NaN   0.86   0.88     0.56

In [51]:
EFFECTS = ["Rain (rainy vs dry)", "Wind (windy vs calm)", "Temperature (warm vs cool)"]

def effect_heatmap(effect):
    base = alt.Chart(city_effect.query("effect == @effect")).encode(
        x=alt.X("pollen_type:N", sort=POLLEN[1:], title=None, axis=alt.Axis(labelAngle=0, orient="top")),
        y=alt.Y("city:N", sort=CITIES, title=None),
    )
    cells = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color("times:Q", title="× pollen",
                        scale=alt.Scale(type="log", domain=[0.25, 1, 4], range=[DIVERGING[0], DIVERGING[2], DIVERGING[4]],
                                        interpolate="rgb", clamp=True)),
        tooltip=["city:N", "pollen_type:N", alt.Tooltip("times:Q", format=".2f", title="× pollen")],
    )
    labels = base.mark_text(fontSize=11).encode(
        text=alt.Text("times:Q", format=".1f"),
        color=alt.condition("datum.times > 3 || datum.times < 0.35", alt.value("white"), alt.value("#0b0b0b")),
    )
    return alt.layer(cells, labels).properties(width=180, height=220, title=title(effect))

(
    alt.hconcat(*[effect_heatmap(e) for e in EFFECTS])
    .resolve_scale(color="shared")
    .properties(title=title("Weather effect per city and pollen type",
                            "Pollen on one kind of day compared with the opposite kind (×) · blue = less, red = more, 2021–2026"))
    .configure_view(stroke=None)
)

alt.HConcatChart(...)

**Answer Q4** (birch, grass and mugwort, compared with the average day of the same season; alder left out because of weak data):
- **Rain lowers pollen for every type in every city.** On rainy days (> 2 mm) pollen is ~**0.6–0.7×** the average and on dry days ~**1.25–1.3×**. Per city, rainy days have **0.3–0.8×** the pollen of dry days — usually **about half**. Rain washes pollen out of the air and keeps flowers closed.
- **Warm days raise pollen for every type in every city** — the strongest effect.
  - **Grass and mugwort:** the warmest third of days have ~**1.3–1.8×** the pollen of the coolest third (only mugwort in Malmö stays flat, 1.0×).
  - **Birch:** ~**1.5–2×** in the south, **~4–5.5×** in the middle and north, where cold spells in May stop the release completely.
- **Wind has a smaller and mixed effect.** For **grass and mugwort**, windy days have less pollen (~**0.5–0.9×**) in almost every city (dilution). For **birch** it is weak (0.7–1.2×).
- **Worst combination:** warm, dry, calm days. **Best:** rainy and cool days.

## Q5. How many days do birch, grass and mugwort take to rise again after rain, per city?

Rain events inside the season: a day with ≥ 2 mm rain after a dry day. Recovery = first day when pollen is back to ≥ 80% of the level before the rain.
Birch, grass and mugwort, all cities together and **per city**.

**Alder is left out:** in the CAMS model alder is near zero outside Göteborg, so there are only ~20 usable rain events (2–7 per city) – too few to trust (weak data). Per city there are also few events for the other types (often < 15), so per-city numbers are indicative.

In [ ]:
d = query("SELECT * FROM marts.daily ORDER BY city, date_local")
s = query("SELECT * FROM marts.seasons WHERE pollen_type IN ('birch', 'grass', 'mugwort')")  # alder left out: weak data

events = []
for _, season in s.iterrows():
    col = f"{season.pollen_type}_pollen_avg"
    c = d[d["city"] == season.city].reset_index(drop=True)
    in_season = c.index[(c["date_local"] >= season.season_start) & (c["date_local"] <= season.season_end)]
    for i in in_season:
        if i == 0 or i + 7 >= len(c):
            continue
        rain_day, day_before = c.loc[i], c.loc[i - 1]
        if rain_day["precipitation_mm"] < 2 or day_before["precipitation_mm"] >= 0.1 or day_before[col] < 1:
            continue
        before = day_before[col]
        recovered = next((k for k in range(1, 8) if c.loc[i + k, col] >= 0.8 * before), None)
        events.append({"pollen_type": season.pollen_type.capitalize(), "city": season.city,
                       "drop": rain_day[col] / before, "recovery_days": recovered})
events = pd.DataFrame(events)

summary = events.groupby("pollen_type").agg(
    events=("drop", "size"),
    pollen_on_rain_day=("drop", "median"),
    median_recovery_days=("recovery_days", "median"),
    recovered_within_2_days=("recovery_days", lambda r: (r <= 2).mean()),
    not_recovered_in_7_days=("recovery_days", lambda r: r.isna().mean()),
).reindex(POLLEN[1:])
summary.round(2)

In [ ]:
curve = pd.DataFrame([
    {"pollen_type": t, "days_after_rain": k, "share": (g["recovery_days"] <= k).mean()}
    for t, g in events.groupby("pollen_type") for k in range(0, 8)
])

(
    alt.Chart(curve)
    .mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=40, filled=True))
    .encode(
        x=alt.X("days_after_rain:O", title="Days after the rain day", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("share:Q", title="Events recovered (≥ 80% of pre-rain level)",
                axis=alt.Axis(format="%", gridColor=GRID), scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("pollen_type:N", sort=POLLEN, scale=POLLEN3_SCALE, title="Pollen"),
        tooltip=["pollen_type:N", "days_after_rain:O", alt.Tooltip("share:Q", format=".0%")],
    )
    .properties(width=520, height=280, title=title("About half of rain events recover within 2 days",
                                                   "Share of rain events where pollen is back to ≥ 80% of the pre-rain level, all cities"))
    .configure_view(stroke=None)
)

In [ ]:
# Per city: share of rain events recovered within 2 days, with the number of events (n)
per_city = (events.groupby(["pollen_type", "city"])
            .agg(n=("drop", "size"), within_2_days=("recovery_days", lambda r: (r <= 2).mean()))
            .reset_index())
per_city["label"] = ((per_city["within_2_days"] * 100).round(0).astype(int).astype(str) + "% (n=" + per_city["n"].astype(str) + ")"
                     + per_city["n"].lt(8).map({True: " ⚠", False: ""}))

base = alt.Chart(per_city).encode(
    x=alt.X("pollen_type:N", sort=POLLEN[1:], title=None, axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("city:N", sort=CITIES, title=None),
)
(
    alt.layer(
        base.mark_rect(stroke="white", strokeWidth=2).encode(
            # grey = fewer than 8 events (too few to trust)
            color=alt.condition("datum.n < 8", alt.value("#e1e0d9"),
                                alt.Color("within_2_days:Q", title="Recovered within 2 days",
                                          scale=alt.Scale(domain=[0, 1], range=BLUE_RAMP), legend=alt.Legend(format="%"))),
            tooltip=["city:N", "pollen_type:N", alt.Tooltip("within_2_days:Q", format=".0%"), "n:Q"],
        ),
        base.mark_text(fontSize=11).encode(
            text="label:N",
            color=alt.condition("datum.within_2_days > 0.6 && datum.n >= 8", alt.value("white"), alt.value("#0b0b0b")),
        ),
    )
    .properties(width=420, height=240,
                title=title("Recovery after rain per city",
                            "Share of rain events back to ≥ 80% within 2 days · n = number of events · grey ⚠ = fewer than 8 events (weak)"))
    .configure_view(stroke=None)
)

**Answer Q5** (birch, grass and mugwort; alder left out because of weak data):
- On the rain day pollen drops to **~45–60%** of the day before.
- After a **single rain day**, pollen is back to ≥ 80% in about **1.5–2 days** (median 1.5 days for birch, 2 days for grass and mugwort).
- About **half of all events recover within 2 days**; about **1 in 3 never recover within a week** (rain continues, or the season is ending).
- **Per city** most cells show **40–70%** recovered within 2 days, but the range is wide (from 12% for mugwort in Luleå to 83% for mugwort in Malmö). The numbers are based on only ~7–26 events per city and type, so differences between cities are not reliable.
- **Tip:** the day **after** a single rainy day is often already a high-pollen day again; a rainy *period* gives longer relief.

## Q6. Which days combine high pollen and bad air quality?

**High pollen day** (daily average, common Nordic level scales; assumption): birch or alder > 100 grains/m³, grass or mugwort > 30.
**Bad air day** (WHO 2021 guidelines): PM2.5 (fine particles smaller than 2.5 µm) daily average > 15 µg/m³ **or** ozone 8-hour average > 100 µg/m³.

In [16]:
risk = query("""
    WITH ozone_8h AS (
        SELECT city, date_local, max(o3_8h) AS ozone_8h_max
        FROM (SELECT city, date_local,
                     avg(ozone) OVER (PARTITION BY city ORDER BY time_utc
                                      ROWS BETWEEN 7 PRECEDING AND CURRENT ROW) AS o3_8h
              FROM marts.hourly)
        GROUP BY ALL
    )
    SELECT d.city, d.date_local, year(d.date_local) AS year,
           d.birch_pollen_avg, d.alder_pollen_avg, d.grass_pollen_avg, d.mugwort_pollen_avg,
           d.pm2_5_avg, o.ozone_8h_max
    FROM marts.daily d
    JOIN ozone_8h o USING (city, date_local)
""")
risk["high_pollen"] = ((risk["birch_pollen_avg"] > 100) | (risk["alder_pollen_avg"] > 100)
                       | (risk["grass_pollen_avg"] > 30) | (risk["mugwort_pollen_avg"] > 30))
risk["bad_pm25"] = risk["pm2_5_avg"] > 15
risk["bad_ozone"] = risk["ozone_8h_max"] > 100
risk["both"] = risk["high_pollen"] & (risk["bad_pm25"] | risk["bad_ozone"])

print(f"High-pollen days: {risk['high_pollen'].sum()} · bad-air days: {(risk['bad_pm25'] | risk['bad_ozone']).sum()} "
      f"· both: {risk['both'].sum()} (ozone: {(risk['both'] & risk['bad_ozone']).sum()}, "
      f"PM2.5: {(risk['both'] & risk['bad_pm25']).sum()})")
risk[risk["both"]].pivot_table(index="city", columns="year", values="both", aggfunc="sum", fill_value=0).reindex(CITIES)

High-pollen days: 659 · bad-air days: 419 · both: 62 (ozone: 59, PM2.5: 5)


year,2021,2022,2023,2024,2025,2026
city,,,,,,
Malmö,4,3,2,1,3,3
Göteborg,4,1,5,0,0,1
Stockholm,5,0,8,0,0,1
Uppsala,3,1,6,0,0,1
Umeå,2,0,5,0,0,0
Luleå,0,0,3,0,0,0


In [17]:
counts = (risk.groupby(["city", "year"], as_index=False)["both"].sum())
heat = alt.Chart(counts).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("city:N", sort=CITIES, title=None),
)
heatmap = alt.layer(
    heat.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color("both:Q", title="Risk days", scale=alt.Scale(range=BLUE_RAMP)),
        tooltip=["city:N", "year:O", alt.Tooltip("both:Q", title="risk days")],
    ),
    heat.mark_text(fontSize=12).encode(
        text="both:Q",
        color=alt.condition("datum.both >= 5", alt.value("white"), alt.value("#0b0b0b")),
    ),
).properties(width=330, height=240, title=title("Risk days per city and year"))

months = (risk[risk["both"]].assign(month=lambda r: pd.to_datetime(r["date_local"]).dt.strftime("%b"),
                                    cause=lambda r: r["bad_ozone"].map({True: "Ozone", False: "PM2.5 (fine particles)"})))
by_month = alt.Chart(months).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4, stroke="white", strokeWidth=2).encode(
    x=alt.X("month:N", sort=["Mar", "Apr", "May", "Jun", "Jul", "Aug"], title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Risk days (all cities, 2021–2026)", axis=alt.Axis(gridColor=GRID)),
    color=alt.Color("cause:N", title="Bad air from", scale=alt.Scale(domain=["Ozone", "PM2.5 (fine particles)"], range=CATEGORICAL[:2])),
    tooltip=["month:N", "cause:N", "count():Q"],
).properties(width=260, height=240, title=title("When do they happen?"))

(
    alt.hconcat(heatmap, by_month)
    .properties(title=title("Days with high pollen AND bad air quality",
                            "High pollen: birch/alder > 100, grass/mugwort > 30 grains/m³ · Bad air: WHO PM2.5 (fine particles) > 15 or ozone 8h > 100 µg/m³"))
    .configure_view(stroke=None)
)

alt.HConcatChart(...)

**Answer Q6:**
- **62 risk days** in 2021–2026 (out of ~660 high-pollen days): roughly **1–2 per city per year**, but up to **8 in Stockholm in 2023**.
- **Ozone is almost always the cause** (59 of 62 days); PM2.5 (fine particles) only 5 times (some days have both).
- They happen mainly in **May** (37 of 62: birch season + the spring ozone peak), then **June** (16, grass) and **April** (8); almost never in July. They fall on **sunny, warm, dry days**: the same weather that releases pollen also forms ozone.
- **2023** was the worst year (warm, dry mid-May in Stockholm, Uppsala, Umeå, Luleå).
- **Tip:** on sunny, warm days in May–June, people with asthma + pollen allergy should be extra careful in the afternoon (ozone and pollen both peak then).

## Q7. Which year had the most intense season for each pollen type and city, and why?

Intensity = **sum of daily pollen over the year** (Seasonal Pollen Integral), shown as an index per city and pollen type: **100 = that city's average year**.

⚠️ **Weak data:** alder is near zero outside Göteborg in the CAMS model, so alder results (except Göteborg) are uncertain.

In [ ]:
intensity = query("SELECT city, pollen_type, year, year_total FROM marts.seasons")
intensity["pollen_type"] = intensity["pollen_type"].str.capitalize()
intensity["index"] = intensity["year_total"] / intensity.groupby(["city", "pollen_type"])["year_total"].transform("mean") * 100

# Average index of the 6 cities, per pollen type and year
intensity.groupby(["pollen_type", "year"])["index"].mean().unstack().round(0).reindex(POLLEN)

In [ ]:
def intensity_panel(pollen):
    base = alt.Chart(intensity.query("pollen_type == @pollen")).encode(
        x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("city:N", sort=CITIES, title=None),
    )
    return alt.layer(
        base.mark_rect(stroke="white", strokeWidth=2).encode(
            color=alt.Color("index:Q", title="Index (100 = average)",
                            scale=alt.Scale(domain=[50, 100, 150], range=[DIVERGING[0], DIVERGING[2], DIVERGING[4]],
                                            interpolate="rgb", clamp=True)),
            tooltip=["city:N", "year:O", alt.Tooltip("index:Q", format=".0f"), alt.Tooltip("year_total:Q", format=",.0f")],
        ),
        base.mark_text(fontSize=11).encode(
            text=alt.Text("index:Q", format=".0f"),
            color=alt.condition("datum.index > 145 || datum.index < 55", alt.value("white"), alt.value("#0b0b0b")),
        ),
    ).properties(width=300, height=200, title=title(pollen))

(
    alt.vconcat(alt.hconcat(intensity_panel("Alder"), intensity_panel("Birch")),
                alt.hconcat(intensity_panel("Grass"), intensity_panel("Mugwort")))
    .resolve_scale(color="shared")
    .properties(title=title("Season intensity per pollen type, city and year",
                            "Index (100 = city average 2021–2026); red = above average, blue = below"))
    .configure_view(stroke=None)
)

In [ ]:
# Why? Compare intensity with the weather that is known to matter for each plant:
# - previous summer (Jun–Jul of the year before): trees (birch, alder) form next year's flowers then
# - spring temperature (Mar–May) and weather during the season itself: matters most for grass and mugwort
weather = query("""
    SELECT city, year(date_local) AS year, month(date_local) AS month,
           temperature_avg_c, precipitation_mm, date_local
    FROM marts.daily
""")
prev_summer = (weather.query("month in [6, 7]").groupby(["city", "year"], as_index=False)["temperature_avg_c"].mean()
               .rename(columns={"temperature_avg_c": "prev_summer_temp"}).assign(year=lambda x: x["year"] + 1))
spring = (weather.query("month in [3, 4, 5]").groupby(["city", "year"], as_index=False)["temperature_avg_c"].mean()
          .rename(columns={"temperature_avg_c": "spring_temp"}))

seasons_all = query("SELECT * FROM marts.seasons")
seasons_all["pollen_type"] = seasons_all["pollen_type"].str.capitalize()
in_season = weather.merge(seasons_all[["city", "pollen_type", "year", "season_start", "season_end"]], on=["city", "year"])
in_season = in_season[(in_season["date_local"] >= in_season["season_start"]) & (in_season["date_local"] <= in_season["season_end"])]
season_weather = in_season.groupby(["city", "pollen_type", "year"], as_index=False).agg(
    season_temp=("temperature_avg_c", "mean"),
    season_rain_days=("precipitation_mm", lambda p: (p >= 1).mean()))

drivers = (intensity.merge(prev_summer, on=["city", "year"], how="left")
           .merge(spring, on=["city", "year"]).merge(season_weather, on=["city", "pollen_type", "year"]))

DRIVERS = {"prev_summer_temp": "Previous summer temperature", "spring_temp": "Spring temperature (Mar–May)",
           "season_temp": "Temperature during season", "season_rain_days": "Rain days during season"}

def spearman(g, col):
    g = g[["index", col]].dropna()
    return g["index"].rank().corr(g[col].rank())   # Spearman = correlation of the ranks

(pd.DataFrame({label: drivers.groupby("pollen_type")[["index", col]].apply(spearman, col=col)
               for col, label in DRIVERS.items()})
   .reindex(POLLEN).round(2))

**Answer Q7:**

| Pollen | Most intense years (6-city average) | Weakest year | Strongest weather link (Spearman) |
|---|---|---|---|
| **Birch** | **2021, 2022** (~116) | 2026 (~83) | Warm **previous summer** (+0.34) |
| **Grass** | **2026** (~118), 2025 (~110) | 2023 (~81) | **Fewer rain days during the season** (−0.43) and a warmer season (+0.34) |
| **Mugwort** | **2026** (~141), 2022 (~131) | 2021 (~55) | Warm previous summer (+0.41), spring temperature (+0.25) |
| **Alder** | **2026** (~201) | 2022 (~16) | Weak and contradictory (weak data) |

- **The "worst year" depends on the pollen type and the city:** e.g. birch peaked in 2021 in Luleå and Umeå, in 2023 in Göteborg and in 2025 in Stockholm; grass peaked in 2025 in Umeå. There is no single national "worst year".
- **Why – different plants, different drivers:**
  - **Birch (tree):** flowers form the summer before → a **warm previous summer** goes with a more intense season.
  - **Grass:** depends on the **same season's weather** → a **dry, warm June–July** gives more pollen; a rainy season gives less.
  - **Mugwort** also follows warm conditions; **alder** shows no clear pattern (very low levels outside Göteborg).
- All correlations are **weak to moderate** (|ρ| ≤ 0.43, ~25–36 city-years per type): only 6 years, natural two-year rhythms in trees and model estimates (CAMS). → Compare with measured data (CAMM, Stockholm 1973–2024) to confirm the "why".

## Allergy risk calendar: which weeks are safest for outdoor activities?

Pollen only (all four types together), per city and ISO week (March–September, 2021–2026):
- **High pollen day:** birch or alder > 100, grass or mugwort > 30 grains/m³ (strong symptoms likely).
- **Moderate or high pollen day:** any pollen type > 10 grains/m³ (sensitive people can notice it).

In [ ]:
allergy = query("SELECT * FROM marts.daily")
allergy["week"] = pd.to_datetime(allergy["date_local"]).dt.isocalendar().week.astype(int)
allergy = allergy[allergy["week"].between(10, 38)].copy()

allergy["High pollen"] = ((allergy["birch_pollen_avg"] > 100) | (allergy["alder_pollen_avg"] > 100)
                          | (allergy["grass_pollen_avg"] > 30) | (allergy["mugwort_pollen_avg"] > 30))
allergy["Moderate or high pollen"] = ((allergy["birch_pollen_avg"] > 10) | (allergy["alder_pollen_avg"] > 10)
                                      | (allergy["grass_pollen_avg"] > 10) | (allergy["mugwort_pollen_avg"] > 10))

allergy_week = (allergy.groupby(["city", "week"], as_index=False)[["High pollen", "Moderate or high pollen"]].mean()
                .melt(id_vars=["city", "week"], var_name="level", value_name="share"))

# Weeks with practically no high-pollen days (< 5%) in each city, between April and August (weeks 14–35)
safe = allergy_week.query("level == 'High pollen' and 14 <= week <= 35 and share < 0.05")
def as_ranges(weeks):
    """[14, 20, 21, 22] -> '14, 20–22'"""
    groups = []
    for w in sorted(weeks):
        if groups and w == groups[-1][-1] + 1:
            groups[-1].append(w)       # consecutive week: extend the current group
        else:
            groups.append([w])         # gap: start a new group
    return ", ".join(str(g[0]) if len(g) == 1 else f"{g[0]}–{g[-1]}" for g in groups)


safe.groupby("city")["week"].apply(lambda w: as_ranges(list(w))).reindex(CITIES).rename("weeks with < 5% high-pollen days")

In [ ]:
def allergy_panel(level):
    return (
        alt.Chart(allergy_week.query("level == @level"))
        .mark_rect(stroke="white", strokeWidth=1)
        .encode(
            x=alt.X("week:O", title="Week of the year", axis=alt.Axis(labelAngle=0)),
            y=alt.Y("city:N", sort=CITIES, title=None),
            color=alt.Color("share:Q", title="Share of days", scale=alt.Scale(domain=[0, 1], range=BLUE_RAMP),
                            legend=alt.Legend(format="%")),
            tooltip=["city:N", "week:O", alt.Tooltip("share:Q", format=".0%", title="share of days")],
        )
        .properties(width=700, height=150, title=title(level))
    )

(
    alt.vconcat(allergy_panel("High pollen"), allergy_panel("Moderate or high pollen"))
    .properties(title=title("Allergy risk calendar: pollen risk per week",
                            "Share of days 2021–2026 per city and ISO week (week 10 ≈ early March, week 38 ≈ mid September) · lighter = safer"))
    .configure_view(stroke=None)
)

**Allergy risk calendar – what it shows:**
- **Highest risk:** weeks **16–19** in the south and middle (birch, mid-April to mid-May), weeks **20–22** in the north (birch, late May), and weeks **24–27** in the south and Uppsala (grass, mid-June to early July). In March (weeks 10–13) only alder, mainly in Göteborg.
- **Safest windows:**
  - **Week 23 (early June):** practically no high-pollen days anywhere (~1%) – the gap between birch and grass. In the north moderate levels are still common (~40–50%).
  - **From week 31 (August):** no pollen risk in any city.
  - **North (Umeå, Luleå) in April** and **south (Malmö, Göteborg) in late May**, when birch is over there.
  - **Stockholm** is the safest city in summer: no high-pollen days from week 22.
- **Tip:** combine with Q3 (early morning is best) and Q4–Q5 (the day after a single rainy day pollen is often back).

## Business questions (Q8–Q12)

Questions for companies that depend on pollen: **pharmacies** (Q8), **tourism** (Q9), **outdoor events** (Q10) **home & retail** (Q11) and a **retail A/B test design** (Q12).

Definitions used in Q9–Q11 (same as Q6): **high pollen day** = birch or alder > 100, grass or mugwort > 30 grains/m³ (daily average). For sensitive people we also use a **moderate pollen day** = any pollen type > 10 grains/m³.

## Q8. In which weeks should pharmacies stock up on antihistamines, per pollen type and city?

For each city and week: **share of years when that week was inside the season**, for all four pollen types.
Recommendation: **stock ready 1 week before the earliest typical start** (10th percentile of start weeks), keep high stock **until the typical end**.

⚠️ **Weak data for alder and mugwort:** in the CAMS model alder is near zero outside Göteborg, and mugwort has low levels and the same dates in every city. **Birch and grass drive most of the antihistamine demand**; use alder and mugwort only as a guide.

In [ ]:
seasons_all = query("SELECT * FROM marts.seasons")
seasons_all["pollen_type"] = seasons_all["pollen_type"].str.capitalize()

weeks = pd.DataFrame({"week": range(5, 37)})
coverage = (seasons_all.merge(weeks, how="cross")
            .assign(in_season=lambda x: (x["week"] >= x["start_week"]) & (x["week"] <= x["end_week"]))
            .groupby(["city", "pollen_type", "week"], as_index=False)["in_season"].mean())

plan = (seasons_all.groupby(["pollen_type", "city"])
        .agg(earliest_start=("start_week", lambda w: w.quantile(0.1)),
             typical_start=("start_week", "median"), typical_peak=("peak_week", "median"),
             typical_end=("end_week", "median"), yearly_pollen=("year_total", "mean"))
        .reset_index())
plan["stock_ready_by_week"] = (plan["earliest_start"] - 1).round().astype(int)
for col in ["typical_start", "typical_peak", "typical_end"]:
    plan[col] = plan[col].round().astype(int)
plan["data_quality"] = plan["pollen_type"].map({"Birch": "good", "Grass": "good", "Alder": "weak", "Mugwort": "weak"})
plan.loc[(plan["pollen_type"] == "Alder") & (plan["city"] == "Göteborg"), "data_quality"] = "ok"

# Stock-ready week per city (rows) and pollen type (columns)
plan.pivot(index="city", columns="pollen_type", values="stock_ready_by_week").reindex(index=CITIES, columns=POLLEN)

In [ ]:
def planning_panel(pollen):
    grid = alt.Chart(coverage.query("pollen_type == @pollen")).mark_rect(stroke="white", strokeWidth=1).encode(
        x=alt.X("week:O", title="Week of the year", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("city:N", sort=CITIES, title=None),
        color=alt.Color("in_season:Q", title="Share of years in season",
                        scale=alt.Scale(domain=[0, 1], range=BLUE_RAMP), legend=alt.Legend(format="%")),
        tooltip=["city:N", "week:O", alt.Tooltip("in_season:Q", format=".0%", title="years in season")],
    )
    ready = alt.Chart(plan.query("pollen_type == @pollen")).mark_text(
        text="▼", fontSize=13, color="#e34948", dy=-9).encode(
        x=alt.X("stock_ready_by_week:O"), y=alt.Y("city:N", sort=CITIES),
        tooltip=["city:N", alt.Tooltip("stock_ready_by_week:O", title="stock ready by week"), "data_quality:N"],
    )
    quality = {"Alder": " (weak data except Göteborg)", "Mugwort": " (weak data)"}.get(pollen, "")
    return alt.layer(grid, ready).properties(width=680, height=150, title=title(pollen + quality))

(
    alt.vconcat(*[planning_panel(p) for p in POLLEN])
    .resolve_scale(x="shared")
    .properties(title=title("Pharmacy planning: when is each city in season, per pollen type?",
                            "Colour = share of years 2021–2026 the week was in season · red ▼ = have stock ready by this week"))
    .configure_view(stroke=None)
)

**Answer Q8** — have stock ready by these ISO weeks (1 week before the earliest typical start):

| City | Alder ⚠️ | **Birch** | **Grass** | Mugwort ⚠️ |
|---|---|---|---|---|
| Malmö | 6 | **13** | **20** | 26 |
| Göteborg | 6 (ok data) | **14** | **20** | 26 |
| Stockholm | 8 | **14** | **21** | 26 |
| Uppsala | 8 | **14** | **22** | 27 |
| Umeå | 9 | **16** | **23** | 26 |
| Luleå | 8 | **16** | **22** | 26 |

- **Priority = birch and grass** (reliable data, most pollen). Birch is by far the largest: yearly pollen ~3.4K–7.9K vs ~0.2K–1K for grass. The biggest birch seasons are in **Göteborg, Uppsala and Umeå**; grass is strongest in **Malmö, Uppsala and Göteborg**.
- **Two demand waves per year:** birch (stock ready week **13–14** in the south and middle → **16** in the north) and grass (week **20** in Malmö/Göteborg → **22–23** in the north). The highest weekly demand is around the **birch peak (weeks 17–18 south/middle, 20–21 north)**.
- **Roll-out south → north:** about **2–3 weeks** between the south/middle and Umeå/Luleå, for both birch and grass.
- **Alder (⚠️ weak data except Göteborg):** very early (from week 6–9, February–March); only relevant in **Göteborg**, where alder pollen is ~7× higher than in the other cities.
- **Mugwort (⚠️ weak data):** late July in every city (stock ready ~week 26–27); low levels → small extra demand.
- **Keep a buffer for early years:** in warm springs (e.g. 2025) birch started 2–3 weeks earlier than typical everywhere.

In [24]:
# Daily data with pollen / air-quality flags, used in Q9–Q11
days = query("SELECT * FROM marts.daily")
days["date_local"] = pd.to_datetime(days["date_local"])
days["high"] = ((days["birch_pollen_avg"] > 100) | (days["alder_pollen_avg"] > 100)
                | (days["grass_pollen_avg"] > 30) | (days["mugwort_pollen_avg"] > 30))
days["moderate"] = ((days["birch_pollen_avg"] > 10) | (days["alder_pollen_avg"] > 10)
                    | (days["grass_pollen_avg"] > 10) | (days["mugwort_pollen_avg"] > 10))
days["bad_pm25"] = days["pm2_5_avg"] > 15
days["month"] = days["date_local"].dt.month
days["week"] = days["date_local"].dt.isocalendar().week.astype(int)

## Q9. Tourism: which destinations are best in spring for people with allergies?

Share of **high-pollen days** per city in each half-month of spring (April–June), 2021–2026. Lower = better destination.

In [25]:
spring = days[days["month"].isin([4, 5, 6])].copy()
spring["period"] = spring["date_local"].dt.strftime("%b") + spring["date_local"].dt.day.map(lambda d: " 1–15" if d <= 15 else " 16–end")
PERIODS = [f"{m} {h}" for m in ["Apr", "May", "Jun"] for h in ["1–15", "16–end"]]

tourism = spring.groupby(["city", "period"], as_index=False)["high"].mean()
best = tourism.loc[tourism.groupby("period")["high"].idxmin()].set_index("period").reindex(PERIODS)
best.rename(columns={"city": "best_destination", "high": "share_high_days"}).round(2)

,best_destination,share_high_days
period,,
Apr 1–15,Luleå,0.00
Apr 16–end,Luleå,0.00
May 1–15,Luleå,0.08
May 16–end,Malmö,0.02
Jun 1–15,Stockholm,0.00
Jun 16–end,Luleå,0.00


In [26]:
base = alt.Chart(tourism).encode(
    x=alt.X("period:N", sort=PERIODS, title=None, axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("city:N", sort=CITIES, title=None),
)
(
    alt.layer(
        base.mark_rect(stroke="white", strokeWidth=2).encode(
            color=alt.Color("high:Q", title="High-pollen days", scale=alt.Scale(domain=[0, 0.75], range=BLUE_RAMP, clamp=True),
                            legend=alt.Legend(format="%")),
            tooltip=["city:N", "period:N", alt.Tooltip("high:Q", format=".0%", title="high-pollen days")],
        ),
        base.mark_text(fontSize=12).encode(
            text=alt.Text("high:Q", format=".0%"),
            color=alt.condition("datum.high > 0.4", alt.value("white"), alt.value("#0b0b0b")),
        ),
    )
    .properties(width=560, height=260,
                title=title("Spring travel: follow the opposite direction of the birch wave",
                            "Share of high-pollen days per city and half-month, 2021–2026 (lighter = better destination)"))
    .configure_view(stroke=None)
)

alt.LayerChart(...)

**Answer Q9:** the best destination **changes every two weeks**, because the pollen wave moves south → north.

| Travel period | Best destinations | Avoid |
|---|---|---|
| **April** | **Umeå, Luleå** (0–1% high days; birch has not started) | Göteborg (late April: 71%), Malmö |
| **Early May** | **Luleå** (8%), Umeå | Stockholm, Uppsala (~50%) |
| **Late May** | **Malmö, Göteborg** (2–5%; birch is over) | Umeå, Luleå (44–64%) |
| **Early June** | **Everywhere is low** (0–10%) – best period for travel | – |
| **Late June** | **Stockholm, Umeå, Luleå** (0%) | Malmö (38%, grass), Uppsala |

- **Rule of thumb:** in April go north, in late May go south; early June is good everywhere.
- Over the whole spring, **Luleå** has the fewest high-pollen days (11%) and **Göteborg** the most (27%).

## Q10. Outdoor events: which dates have the lowest risk of high pollen?

For a festival or a race, three things can spoil the day. Each is measured as the **share of days** per ISO week (April–September, 2021–2026):
- **Pollen** – per type: alder, birch, grass and mugwort with a daily average > 10 grains/m³ (moderate or high, relevant for sensitive people).
- **Rain** – at least 1 mm during the day.
- **Bad air** – PM2.5 (fine particles smaller than 2.5 µm) > 15 µg/m³ or ozone 8-hour average > 100 µg/m³ (WHO limits).

Heat (max temperature ≥ 25 °C) is shown as extra information for sports events.

In [ ]:
events = query("""
    WITH ozone_8h AS (
        SELECT city, date_local, max(o3_8h) AS ozone_8h_max
        FROM (SELECT city, date_local,
                     avg(ozone) OVER (PARTITION BY city ORDER BY time_utc
                                      ROWS BETWEEN 7 PRECEDING AND CURRENT ROW) AS o3_8h
              FROM marts.hourly)
        GROUP BY ALL
    )
    SELECT d.*, o.ozone_8h_max
    FROM marts.daily d
    JOIN ozone_8h o USING (city, date_local)
""")
events["week"] = pd.to_datetime(events["date_local"]).dt.isocalendar().week.astype(int)
events = events[events["week"].between(14, 38)].copy()

for p in POLLEN:                                            # moderate or high pollen, per type
    events[p] = events[f"{p.lower()}_pollen_avg"] > 10
events["Any pollen"] = events[POLLEN].any(axis=1)
events["Rain (≥ 1 mm)"] = events["precipitation_mm"] >= 1
events["Bad air (PM2.5 or ozone)"] = (events["pm2_5_avg"] > 15) | (events["ozone_8h_max"] > 100)
events["Hot (≥ 25 °C)"] = events["temperature_max_c"] >= 25
events["Any problem"] = events["Any pollen"] | events["Rain (≥ 1 mm)"] | events["Bad air (PM2.5 or ozone)"]

FLAGS = POLLEN + ["Any pollen", "Rain (≥ 1 mm)", "Bad air (PM2.5 or ozone)", "Hot (≥ 25 °C)", "Any problem"]
by_week = (events.groupby(["city", "week"], as_index=False)[FLAGS].mean()
           .melt(id_vars=["city", "week"], var_name="factor", value_name="share"))

# Best 3 weeks per city in the main event season (May–August = weeks 18–35): fewest problem days
best_weeks = (by_week.query("factor == 'Any problem' and 18 <= week <= 35")
              .sort_values("share").groupby("city").head(3)
              .assign(city=lambda x: pd.Categorical(x["city"], CITIES, ordered=True))
              .sort_values(["city", "share"]))
best_weeks.assign(share=lambda x: (x["share"] * 100).round(0).astype(int).astype(str) + "% problem days")[["city", "week", "share"]]

In [ ]:
# 1) Pollen per type: one heatmap per pollen type (city × week)
def week_heatmap(data, panel_title, height=120):
    return (
        alt.Chart(data)
        .mark_rect(stroke="white", strokeWidth=1)
        .encode(
            x=alt.X("week:O", title="Week of the year", axis=alt.Axis(labelAngle=0)),
            y=alt.Y("city:N", sort=CITIES, title=None),
            color=alt.Color("share:Q", title="Share of days", scale=alt.Scale(domain=[0, 1], range=BLUE_RAMP),
                            legend=alt.Legend(format="%")),
            tooltip=["city:N", "week:O", alt.Tooltip("share:Q", format=".0%", title="share of days")],
        )
        .properties(width=620, height=height, title=title(panel_title))
    )

(
    alt.vconcat(*[week_heatmap(by_week.query("factor == @p"), f"{p} pollen > 10 grains/m³") for p in POLLEN])
    .properties(title=title("Pollen risk per type, city and week",
                            "Share of days 2021–2026 with moderate or high pollen (week 14 ≈ early April, week 38 ≈ mid September)"))
    .configure_view(stroke=None)
)

In [ ]:
# 2) Weather and air: average of the 6 cities per week (lines; all shares, so one y-axis)
WEATHER = ["Rain (≥ 1 mm)", "Bad air (PM2.5 or ozone)", "Hot (≥ 25 °C)"]
weather_week = by_week[by_week["factor"].isin(WEATHER)].groupby(["factor", "week"], as_index=False)["share"].mean()

(
    alt.Chart(weather_week)
    .mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=30, filled=True))
    .encode(
        x=alt.X("week:Q", title="Week of the year", scale=alt.Scale(domain=[14, 38]), axis=alt.Axis(tickCount=12)),
        y=alt.Y("share:Q", title="Share of days (6-city average)", axis=alt.Axis(format="%", gridColor=GRID)),
        color=alt.Color("factor:N", title=None, sort=WEATHER, legend=alt.Legend(orient="top"),
                        scale=alt.Scale(domain=WEATHER, range=[CATEGORICAL[0], CATEGORICAL[4], CATEGORICAL[3]])),
        tooltip=["factor:N", "week:Q", alt.Tooltip("share:Q", format=".0%")],
    )
    .properties(width=620, height=240, title=title("Rain is the most frequent spoiler; bad air is rare",
                                                   "Share of days per week, average of the 6 cities, 2021–2026"))
    .configure_view(stroke=None)
)

In [ ]:
# 3) Combined: share of days with any problem (pollen, rain or bad air) – lower = better event date
combined = by_week.query("factor == 'Any problem'")
base = alt.Chart(combined).encode(
    x=alt.X("week:O", title="Week of the year", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("city:N", sort=CITIES, title=None),
)
(
    alt.layer(
        base.mark_rect(stroke="white", strokeWidth=1).encode(
            color=alt.Color("share:Q", title="Problem days", scale=alt.Scale(domain=[0, 1], range=BLUE_RAMP),
                            legend=alt.Legend(format="%")),
            tooltip=["city:N", "week:O", alt.Tooltip("share:Q", format=".0%", title="problem days")],
        ),
        base.mark_text(fontSize=9).encode(
            text=alt.Text("share:Q", format=".0%"),
            color=alt.condition("datum.share > 0.6", alt.value("white"), alt.value("#0b0b0b")),
        ),
    )
    .properties(width=720, height=220,
                title=title("Event planning: share of days with any problem, per city and week",
                            "Pollen (any type) > 10 grains/m³, rain ≥ 1 mm or bad air (WHO limits) · lighter = better date"))
    .configure_view(stroke=None)
)

**Answer Q10:**
- **Pollen depends on the type** (6-city averages, share of days > 10 grains/m³):
  - **Alder:** flowers mostly before April (February–March, see Q1), so it barely affects the event season; a little in early April in Göteborg.
  - **Birch:** weeks 15–22, peak **week 19** (mid May, ~80% of days); weeks 20–22 in the north.
  - **Grass:** weeks 24–29, peak **week 26** (late June, ~65%); strongest in Malmö, Göteborg and Uppsala.
  - **Mugwort:** small, weeks 29–31 (late July).
- **Rain is the most frequent spoiler:** 25–58% of days have ≥ 1 mm; the wettest weeks are **27 (early July)** and **31 (early August)**.
- **Bad air is rare** (< 8% of days), except **week 19 (mid May, ~15%)** when ozone peaks together with birch. **Heat ≥ 25 °C** is most common in weeks 24–33 (up to 25% of days).
- **Best dates for outdoor events (fewest problem days):**
  - **Late August – early September (weeks 33–36):** the best window in every city (week 35: ~17–33% problem days). Pollen is over; only rain remains.
  - **Early June (week 23):** a pollen gap between birch and grass in the south and middle (~50–70% problem days, mainly rain).
  - **North (Umeå, Luleå):** **late April / early May (weeks 16–18)** before birch starts, and **mid-July (weeks 29–30) in Luleå**.
- **Worst dates:** **mid May (week 19)** in the south and middle (birch + ozone) and **early July (week 27)** almost everywhere (grass + rain).

**Why two different "best dates"?** The answer depends on what counts as a bad day:

| Goal | What counts as a bad day | Best date |
|---|---|---|
| **People with allergies** (see the *Allergy risk calendar*) | **High pollen** only | **Early June (week 23)**: ~1% high-pollen days; or August onwards |
| **Any outdoor event** (festival, race) | **Moderate pollen (> 10) + rain + bad air** | **Late August – early September (weeks 33–36)**: 17–33% problem days |

Early June has almost no **high** pollen, but it still has some **moderate** pollen (birch ending, grass starting) and rain on ~37% of days. In late August pollen is **zero** and rain is lower (~26% in week 35), so it wins when everything is counted. The data is the same; only the definition of a "bad day" changes.

## Q11. Home & retail: when to sell air purifiers and filters?

Two reasons to buy an air purifier or filter: **pollen** (high-pollen days) and **fine particles** — days with **PM2.5 (fine particles smaller than 2.5 µm, e.g. from wood burning, traffic and road dust)** above the WHO limit.
Average number of days **per city and month** (2021–2026), so each city can be read on its own row.

In [29]:
# Average number of days per month for each city (share of days × 30.4 days per month)
need = days.groupby(["city", "month"], as_index=False)[["high", "bad_pm25"]].mean()
need["high_days"] = need["high"] * 30.4
need["pm25_days"] = need["bad_pm25"] * 30.4
need["month_name"] = pd.to_datetime(need["month"], format="%m").dt.strftime("%b")
MONTHS = list(pd.to_datetime(range(1, 13), format="%m").strftime("%b"))

# Days per year above the WHO limit for PM2.5 (fine particles), per city
display((days.groupby("city")["bad_pm25"].mean() * 365).round(1).reindex(CITIES)
        .rename("PM2.5 (fine particles) days over WHO limit per year"))

# Average of the 6 cities, per month
need.groupby("month_name")[["high_days", "pm25_days"]].mean().reindex(MONTHS).round(1)

city
Malmö        21.2
Göteborg      8.9
Stockholm     4.0
Uppsala       3.1
Umeå          0.2
Luleå         0.9
Name: PM2.5 days over WHO limit per year, dtype: float64

,month_name,high_days,pm25_days
0,Jan,0.0,0.4
1,Feb,0.0,1.0
2,Mar,0.1,2.2
3,Apr,6.0,0.3
4,May,8.6,0.0
5,Jun,2.7,0.1
6,Jul,0.7,0.1
7,Aug,0.0,0.1
8,Sept,0.0,0.6
9,Oct,0.0,0.5


In [30]:
def city_month_heatmap(field, chart_title, legend_title):
    base = alt.Chart(need).encode(
        x=alt.X("month_name:N", sort=MONTHS, title=None, axis=alt.Axis(labelAngle=0, orient="top")),
        y=alt.Y("city:N", sort=CITIES, title=None),
    )
    cells = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color(f"{field}:Q", title=legend_title, scale=alt.Scale(range=BLUE_RAMP)),
        tooltip=["city:N", alt.Tooltip("month_name:N", title="month"), alt.Tooltip(f"{field}:Q", format=".1f", title="days per month")],
    )
    labels = base.mark_text(fontSize=11).encode(
        # show a number only where there is at least half a day; hide the zeros
        text=alt.condition(f"datum.{field} >= 0.5", alt.Text(f"{field}:Q", format=".0f"), alt.value("")),
        color=alt.condition(f"datum.{field} >= 8", alt.value("white"), alt.value("#0b0b0b")),
    )
    return alt.layer(cells, labels).properties(width=560, height=200, title=title(chart_title))

(
    alt.vconcat(
        city_month_heatmap("high_days", "Pollen: high-pollen days per month", "Days / month"),
        city_month_heatmap("pm25_days", "Fine particles: days with PM2.5 (fine particles) above the WHO limit per month",
                           "Days / month"),
    )
    .resolve_scale(color="independent")
    .properties(title=title("Air purifier demand per city: pollen in spring, particles in late winter",
                            "Average number of days per month and city, 2021–2026 (empty cell = less than half a day)"))
    .configure_view(stroke=None)
)

alt.VConcatChart(...)

**Answer Q11:**
- **Main sales season: March → June (pollen).** High-pollen days peak in **April in the south** (Göteborg ~15 days, Malmö ~7) and in **May in the middle and north** (Uppsala ~10, Umeå ~13, Luleå ~8). Malmö and Uppsala have a second grass peak in **June–July**.
- **Second window: February–March (fine particles).** Days with **PM2.5 (fine particles smaller than 2.5 µm)** above the WHO limit are most frequent in **March** — **Malmö ~6 days**, Göteborg ~4 — mainly from wood burning, road dust after winter and pollution transported from the continent. **Malmö** also has autumn–winter days (Sep–Dec, ~2–3 per month).
- **The north (Umeå, Luleå) almost never exceeds the PM2.5 limit** → sell purifiers there only for pollen (May).
- **Timeline for a retailer:**
  1. **February–March:** particle filters + purifiers in **Malmö and Göteborg**; start the pollen campaign in the **south** at the end of March.
  2. **April:** pollen peak in the **south** (Göteborg, Malmö) → main push there.
  3. **May:** pollen peak in the **middle and north** (Stockholm, Uppsala, Umeå, Luleå) → move the campaign north; include **car cabin pollen filters**.
  4. **June–July:** grass season in **Malmö and Uppsala** → keep stock; wind down from August.

## Q12. Retail A/B test design: does a campaign that follows the pollen sell more than one that follows the calendar?

Retail is where most A/B tests happen in real life, and pollen drives sales far beyond pharmacies: tissues, eye drops, nasal sprays and antihistamines in supermarkets, air purifiers and filters in electronics/home stores, allergy and skin-care products online, sunglasses and masks in outdoor shops.

| Test | Group A | Group B | Primary metric |
|---|---|---|---|
| Pollen alert email | Normal newsletter | "Birch season is coming" email with allergy products | Allergy-product sales per customer |
| Website banner | Normal banner | Allergy banner on high-pollen days in the customer's city | Conversion rate |
| **Campaign timing** | Campaign on a **fixed date** every year | Campaign starts when the **pollen season starts in the region** | Total allergy-product sales in the season |
| In-store placement | Products in their usual aisle | Products next to the checkout in high-pollen weeks | Sales per store |

**This design focuses on "Campaign timing"**: it connects the analysis (when the season starts in each region) directly to a business decision.
The project has **no sales data**, so this section delivers the **test design**, plus a check with the pollen data that the idea makes sense.

### Step 1 – A trigger that works in real time

The season start used in Q1–Q8 (day when 5% of the year's pollen has fallen) is only known **after** the season. A campaign needs a rule that fires **on the day**:

> **Trigger:** first day of **3 consecutive days with birch > 10 grains/m³** (daily average), from 1 February.

In production the same rule would run on the pollen **forecast** (CAMS gives 4 days ahead) via the daily pipeline. Below: how close the trigger and a fixed date (1 April, same for the whole country) get to the real season start.

In [ ]:
birch_daily = query("SELECT city, date_local, birch_pollen_avg FROM marts.daily ORDER BY city, date_local")
birch_daily["date_local"] = pd.to_datetime(birch_daily["date_local"])
birch_daily["year"] = birch_daily["date_local"].dt.year

def first_trigger(g, threshold=10, run=3):
    """First day of the first run of `run` consecutive days above `threshold`."""
    above = (g["birch_pollen_avg"] > threshold).astype(int).rolling(run).sum()
    hits = g.loc[above == run, "date_local"]
    return hits.iloc[0] - pd.Timedelta(days=run - 1) if len(hits) else pd.NaT

triggers = (birch_daily[birch_daily["date_local"].dt.month >= 2]
            .groupby(["city", "year"]).apply(first_trigger).rename("trigger").reset_index())

timing = triggers.merge(query("SELECT city, year, season_start FROM marts.seasons WHERE pollen_type = 'birch'"),
                        on=["city", "year"])
timing["season_start"] = pd.to_datetime(timing["season_start"])
timing["fixed_date"] = pd.to_datetime(timing["year"].astype(str) + "-04-01")

# Days between each campaign start and the real season start (negative = too early, positive = too late)
timing["Pollen trigger"] = (timing["trigger"] - timing["season_start"]).dt.days
timing["Fixed date (1 April)"] = (timing["fixed_date"] - timing["season_start"]).dt.days

compare = timing.melt(id_vars=["city", "year"], value_vars=["Pollen trigger", "Fixed date (1 April)"],
                      var_name="campaign_start", value_name="days_vs_season")
(compare.groupby(["campaign_start", "city"])["days_vs_season"].median()
        .unstack(0).reindex(CITIES)
        .rename_axis(columns="median days vs real season start"))

In [ ]:
METHODS = ["Pollen trigger", "Fixed date (1 April)"]
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color="#898781", strokeDash=[4, 4]).encode(x="x:Q")
zero_label = alt.Chart(pd.DataFrame({"x": [0], "t": ["real season start"]})).mark_text(
    dy=-8, dx=4, align="left", fontSize=11, color="#52514e").encode(x="x:Q", y=alt.value(0), text="t:N")
dots = alt.Chart(compare).mark_circle(size=70, opacity=0.85, stroke="white", strokeWidth=1).encode(
    x=alt.X("days_vs_season:Q", title="Days vs the real season start (negative = campaign too early)",
            scale=alt.Scale(domain=[-55, 30]), axis=alt.Axis(gridColor=GRID)),
    y=alt.Y("city:N", sort=CITIES, title=None),
    yOffset=alt.YOffset("campaign_start:N", sort=METHODS),
    color=alt.Color("campaign_start:N", sort=METHODS, title="Campaign starts",
                    scale=alt.Scale(domain=METHODS, range=CATEGORICAL[:2]), legend=alt.Legend(orient="top")),
    tooltip=["city:N", "year:O", "campaign_start:N", alt.Tooltip("days_vs_season:Q", title="days vs season start")],
)

(
    alt.layer(zero, zero_label, dots)
    .properties(width=620, height=300,
                title=title("A pollen trigger starts the campaign on time; a fixed date is weeks too early in the north",
                            "Birch, one dot per city and year 2021–2026 · 0 = real season start (5% of the year's pollen)"))
    .configure_view(stroke=None)
)

**What this shows:** the pollen trigger fires **~0–4 days before** the real birch season in almost every city and year (median error **~2 days**). A fixed national date (1 April) is on average **~2 weeks too early** in the south and middle and **~6 weeks too early** in the north (median error **~19 days**), and it moves by up to 3–4 weeks from year to year.
**Risk of the trigger:** in one year in Umeå and Luleå it fired ~3 weeks **late** (pollen stayed just under the threshold for a few days at the start) → in practice combine it with a safety rule, e.g. "start at the latest on the typical start date of the region".
→ A campaign that follows the pollen should reach customers **when they start to need the products**, especially in the north. That is the hypothesis to test.
*(For grass, the same rule fires ~10 days late and sometimes not at all in the north → it needs a lower threshold before use.)*

### Step 2 – Test design

| Element | Design |
|---|---|
| **Hypothesis** | H0: starting the allergy campaign with the pollen trigger gives the same allergy-product sales as the fixed date. H1: the pollen-triggered campaign sells more in the season. |
| **Groups** | **A (control):** campaign starts on a fixed date (1 April) in every store. **B (treatment):** campaign starts on the day the pollen trigger fires in the store's city. Same campaign content and budget in both. |
| **Unit of assignment** | **Stores (or local areas), not customers.** The season reaches a whole region at once, and local marketing (posters, local ads, in-store displays) cannot be shown to some customers and not to others → this is a **geo-experiment**. |
| **Randomization** | **Stratified by region** (South / Middle / North): within each region, stores are paired by size and past allergy sales, and one store of each pair goes to A, the other to B. This keeps both groups balanced in climate and season timing. |
| **Contamination** | Stores in the same small town may share customers → assign by **town / kommun cluster** where stores are close together; keep online campaigns (email, web) out of the test or run them as a separate customer-level test. |
| **Duration** | One full birch season: from **week 6** (earliest possible trigger) to **week 23** (end of birch in the north). Optionally a second run for grass (weeks 20–32) once its trigger is calibrated. |
| **Primary metric** | **Allergy-product sales per store per week** over the season (antihistamines, nasal sprays, eye drops, tissues). |
| **Secondary & guardrails** | Campaign ROI, stock-outs of allergy products, **total store sales** (guardrail: the campaign should not just move sales from other categories). |
| **Analysis** | **Difference-in-differences:** compare B vs A this season, adjusted for each store's sales in the same weeks last year (pre-period). Standard errors clustered by store / cluster. Report the effect per region (expected to be largest in the north, where the fixed date is furthest off). |
| **Data needed** | Weekly sales per store and product category (this year + last year), store location (city / kommun), campaign calendar per store, and the daily pollen trigger from this pipeline. |

### Step 3 – How many stores are needed?

Rough power calculation for comparing the average store result in A vs B. Assumptions (to replace with real data): a store's seasonal allergy sales vary by ±30% between stores (coefficient of variation 0.3), 5% significance level, 80% power. Adjusting for last year's sales (pre-period correlation ~0.7) removes about half of that variation.

In [ ]:
Z_ALPHA, Z_POWER = 1.96, 0.84   # two-sided 5% significance, 80% power

def stores_per_group(lift, cv=0.30, pre_period_corr=0.0):
    """Stores needed per group to detect a relative `lift` in average sales per store.
    pre_period_corr > 0 = variance reduction from adjusting for last year's sales (CUPED-style)."""
    variance = cv ** 2 * (1 - pre_period_corr ** 2)
    return int(round(2 * (Z_ALPHA + Z_POWER) ** 2 * variance / lift ** 2))

pd.DataFrame(
    [{"detectable lift": f"{lift:.0%}",
      "stores per group (no adjustment)": stores_per_group(lift),
      "stores per group (adjusted for last year)": stores_per_group(lift, pre_period_corr=0.7)}
     for lift in [0.03, 0.05, 0.10, 0.15]]
)

**Answer Q12:**
- **The idea is supported by the pollen data:** a real-time pollen trigger (3 days with birch > 10 grains/m³) starts the campaign within **~2 days** of the real season, while a fixed 1 April date is **~2 weeks early in the south and middle and ~6 weeks early in the north**.
- **Test design:** a **geo-experiment** with **stores (or local clusters) as the unit**, **stratified by region** and paired by size; control = fixed date, treatment = pollen trigger; primary metric = allergy-product sales per store-week; analysed with **difference-in-differences** against last year.
- **Size:** to detect a **10% lift** you need roughly **~140 stores per group** without adjustment or **~70** when adjusting for last year's sales; a **5% lift** needs ~290 per group even with adjustment → realistic for a **large national chain**, or run the test over **two seasons**.
- **Why not a normal A/B test:** the season reaches a whole region at once and local marketing cannot be randomized per customer → randomizing customers would mix treated and untreated people in the same store. Saying this in the design shows the **limits of A/B testing** are understood.
- **Expected result:** the largest gain in the **north**, where the fixed date is furthest from the real season.

## Validation: CAMS model vs measured pollen in Stockholm (CAMM)

Our pollen data are **model estimates** (CAMS). To check them, we compare Stockholm with **real measurements** from the CAMM report:

> Lõhmus Sundström M, Stucki L, Andersson N, Ekebom A, Gedda B, Jonsson M et al. *Förändringar i pollensäsonger, klimat och folkhälsa.* Centrum för arbets- och miljömedicin (CAMM), Region Stockholm, 2026. Daily pollen counts from the Palynological Laboratory, Swedish Museum of Natural History, Stockholm, 1973–2024 (alder 1998–2024). Table B1, p. 30. [Report (PDF)](https://www.folkhalsoguiden.se/49c72d/globalassets/verksamheter/forskning-och-utveckling/centrum-for-arbets--och-miljomedicin1/rapporter/pollenrapport_klar.pdf)

**How the comparison is made fair:**
- **Same season definition as CAMM:** start = day when **3%** of the year's pollen has fallen, end = **97%**, peak = day with the highest daily level (we recompute ours with 3% / 97% instead of our usual 5% / 95%).
- **Same period:** CAMM reports the **average over 1973–2024 plus a linear trend per year**. Seasons have moved earlier, so we use the trend to estimate the CAMM value for **2023** (middle of our 2021–2026 period): value 2023 = average + trend per year × (2023 − middle year of the series).
- Dates are shown as day of the year on a normal (non-leap) calendar.

In [ ]:
# CAMM, Table B1 (p. 30): average day of year (DOY), trend per year and annual pollen sum (SPIn, grains/m³)
camm = pd.DataFrame([
    # pollen,   first, last, start,  start_trend, peak,   peak_trend, end,    end_trend, spin,    spin_trend
    ("Alder",   1998, 2024,  69.83,  0.047,       83.56,  0.11,       110.03, 1.17,      1442.33, 35.25),
    ("Birch",   1973, 2024, 117.85, -0.31,       126.23, -0.29,       146.17, -0.14,     8853.00, 78.84),
    ("Grass",   1973, 2024, 156.88, -0.10,       179.71, -0.22,       232.21,  0.24,     1232.20,  8.67),
    ("Mugwort", 1973, 2024, 197.71, -0.12,       216.12,  0.016,      243.23,  0.65,      188.10, -0.08),
], columns=["pollen_type", "first_year", "last_year", "start", "start_trend", "peak", "peak_trend",
            "end", "end_trend", "spin", "spin_trend"])

# Trend-adjusted CAMM value for 2023 (a linear trend passes through the average at the middle year)
years_from_middle = 2023 - (camm["first_year"] + camm["last_year"]) / 2
camm_2023 = pd.DataFrame({"pollen_type": camm["pollen_type"], "source": "CAMM (measured)"})
for col in ["start", "peak", "end", "spin"]:
    camm_2023[col] = camm[col] + camm[f"{col}_trend"] * years_from_middle

# Our CAMS data for Stockholm, recomputed with the CAMM definition (3% / 97% of the year's pollen)
sthlm = query("SELECT * FROM marts.daily WHERE city = 'Stockholm' ORDER BY date_local")
sthlm["date_local"] = pd.to_datetime(sthlm["date_local"])
sthlm["year"] = sthlm["date_local"].dt.year

rows = []
for p in POLLEN:
    col = f"{p.lower()}_pollen_avg"
    for year, g in sthlm.groupby("year"):
        share = g[col].cumsum() / g[col].sum()
        rows.append({"pollen_type": p, "year": year,
                     "start": g.loc[share >= 0.03, "date_local"].iloc[0].dayofyear,
                     "peak": g.loc[g[col].idxmax(), "date_local"].dayofyear,
                     "end": g.loc[share >= 0.97, "date_local"].iloc[0].dayofyear,
                     "spin": g[col].sum()})
cams = (pd.DataFrame(rows).groupby("pollen_type", as_index=False)[["start", "peak", "end", "spin"]].mean()
        .assign(source="CAMS model (this project)"))

validation = pd.concat([camm_2023, cams], ignore_index=True)

def doy_to_date(doy):
    return (pd.Timestamp("2023-01-01") + pd.to_timedelta(round(doy) - 1, unit="D")).strftime("%d %b")

# Side-by-side table: dates, difference in days (CAMS − CAMM) and pollen amount ratio
wide = validation.pivot(index="pollen_type", columns="source").reindex(POLLEN)
table = pd.DataFrame(index=POLLEN)
for col in ["start", "peak", "end"]:
    table[f"{col} CAMM"] = wide[(col, "CAMM (measured)")].map(doy_to_date)
    table[f"{col} CAMS"] = wide[(col, "CAMS model (this project)")].map(doy_to_date)
    table[f"{col} diff (days)"] = (wide[(col, "CAMS model (this project)")] - wide[(col, "CAMM (measured)")]).round(0).astype(int)
table["pollen amount CAMS / CAMM"] = (wide[("spin", "CAMS model (this project)")] / wide[("spin", "CAMM (measured)")]).round(2)
table

In [ ]:
SOURCES = ["CAMM (measured)", "CAMS model (this project)"]
chart_data = validation.assign(
    start_date=lambda d: pd.Timestamp("2023-01-01") + pd.to_timedelta(d["start"].round() - 1, unit="D"),
    peak_date=lambda d: pd.Timestamp("2023-01-01") + pd.to_timedelta(d["peak"].round() - 1, unit="D"),
    end_date=lambda d: pd.Timestamp("2023-01-01") + pd.to_timedelta(d["end"].round() - 1, unit="D"),
)
y = alt.Y("pollen_type:N", sort=POLLEN, title=None)
y_offset = alt.YOffset("source:N", sort=SOURCES)
colour = alt.Color("source:N", sort=SOURCES, title=None, legend=alt.Legend(orient="top"),
                   scale=alt.Scale(domain=SOURCES, range=[CATEGORICAL[1], CATEGORICAL[0]]))
bars = alt.Chart(chart_data).mark_bar(height=12, cornerRadius=4).encode(
    y=y, yOffset=y_offset, color=colour,
    x=alt.X("start_date:T", title=None, scale=alt.Scale(domain=["2023-02-15", "2023-09-30"]),
            axis=alt.Axis(format="%b", tickCount="month", gridColor=GRID)),
    x2="end_date:T",
    tooltip=["pollen_type:N", "source:N", alt.Tooltip("start_date:T", format="%d %b", title="start"),
             alt.Tooltip("peak_date:T", format="%d %b", title="peak"), alt.Tooltip("end_date:T", format="%d %b", title="end")],
)
peaks = alt.Chart(chart_data).mark_point(filled=True, size=60, color="#0b0b0b", stroke="white", strokeWidth=1.5).encode(
    y=y, yOffset=y_offset, x="peak_date:T")

(
    alt.layer(bars, peaks)
    .properties(width=620, height=260,
                title=title("Stockholm: modelled seasons (CAMS) vs measured seasons (CAMM)",
                            "Bar = start (3%) to end (97%), dot = peak · CAMM trend-adjusted to 2023, CAMS = average 2021–2026"))
    .configure_view(stroke=None)
)

**Validation result:**

| Pollen | Season dates (CAMS vs measured) | Pollen amount (CAMS / measured) | Verdict |
|---|---|---|---|
| **Birch** | Start and end within **~1 week**; CAMS peak **~1 week later** | **~0.5×** (model has about half) | ✅ **Timing reliable**, levels underestimated |
| **Grass** | Start and peak within **~1 week**; CAMS season **ends ~2 weeks earlier** | **~0.4×** | ✅ Start and peak reliable; ⚠️ late-summer tail missing |
| **Mugwort** | Start matches; CAMS peak ~1 week earlier and season **ends ~4 weeks earlier** | ~1.3× | ⚠️ Start reliable, end not |
| **Alder** | Dates within **~1 week** (surprisingly close) | **~0.03×** (model has ~3% of the measured amount) | ❌ **Amounts not reliable** → confirms leaving alder out of Q4–Q5 |

*(Exact numbers in the table above.)*

- **Season timing (start and peak) of birch and grass – the two main allergy triggers – is reliable in the model.** This is what Q1, Q2, Q8 and Q12 are built on, so the main business conclusions hold.
- **Absolute pollen levels are too low in the model** (birch ~½, grass ~0.4×). Thresholds like "high pollen > 100 grains/m³" are therefore **reached less often in the model than in reality** → high-pollen days (Q6, risk calendar, Q9–Q11) are **conservative** (real risk is likely higher).
- **Late-summer seasons end too early in the model** (grass, mugwort) → August risk is underestimated.
- **Alder is clearly unreliable** in the model (~3% of the measured amount in Stockholm), which supports the weak-data warnings.
- Caveats: CAMM is **one measuring station** in Stockholm, CAMS is an **~11 km model grid cell**; the CAMM 2023 values are **trend estimates**, not the measured 2021–2024 values; the peak is a single day and varies a lot between years (SD ~8–15 days).

### In simple words

- **CAMS (our data)** is like a **weather forecast**: a computer model that *estimates* how much pollen there should be, for the whole country.
- **CAMM** is like a **thermometer on the wall**: at the Swedish Museum of Natural History in Stockholm, pollen has been caught from the air and counted under a microscope **every day since 1973**. That is the **real** measurement.
- The validation asks: **does our forecast match the real measurement?**

**Why not compare year by year?** The CAMM report does **not publish the yearly dates**, only the **50-year average** and the **trend per year** (e.g. birch starts ~0.31 days earlier every year). So the 2023 value is an **estimate**: average + trend × years. A year-by-year check would need the daily data from Pollenrapporten (charts only, no scraping allowed) or from the laboratory on request.

**Nothing was replaced.** The 3% / 97% definition is used **only in this comparison**, to measure the season the same way as CAMM; Q1–Q12 keep the 5% / 95% definition. CAMM does not replace our data – it is the **referee** that tells us how much to trust each part:
- ✅ **Dates** of birch and grass → trustworthy, the conclusions hold.
- ⚠️ **Amounts** → the model has less pollen than reality, so the real risk is likely **higher**.
- ❌ **Alder** → not trustworthy.

## Recommendations

Summary of the analysis (Q1–Q11) as concrete actions. Based on 6 cities, 2021–2026, CAMS model estimates.
*Pattern information, not medical advice – for treatment, consult your doctor.*

### For people with allergies

| # | Recommendation | Based on |
|---|---|---|
| 1 | **Prepare 1–2 weeks before your city's season.** Birch starts around **mid April** in the south and middle (Malmö, Göteborg, Stockholm, Uppsala) and around **11 May** in the north (Umeå, Luleå), and peaks ~2–3 weeks after the start in the south and middle, but only ~1 week after in the north (short, intense season). In warm springs (e.g. 2025) it can come **2–3 weeks earlier**. Grass starts in early–mid June. | Q1, Q2 |
| 2 | **Be outdoors early in the morning (05:00–08:00).** Pollen is lowest then for all types in all cities and highest in the afternoon/evening. Air the home in the morning, not in the evening. | Q3 |
| 3 | **Use the weather forecast.** Rainy and cool days have about **half** the pollen; warm, dry, calm days are the worst (birch up to 4–5× higher on the warmest days in the middle and north). After a single rainy day, pollen is back within **1.5–2 days**, so plan outdoor activities *on* the rainy day or right after, not two days later. | Q4, Q5 |
| 4 | **Asthma + pollen allergy: be careful on sunny, warm days in May–June.** 62 days in 6 years combined high pollen and bad air, almost always **ozone** (59 of 62), mostly in **May**. | Q6 |
| 5 | **Use the allergy risk calendar to plan trips and holidays.** In April the north is pollen-free; in late May the south is; **early June (week 23)** is low everywhere, and from **August** there is no pollen risk. | Risk calendar, Q9 |

### For businesses

| Sector | Recommendation | Based on |
|---|---|---|
| **Pharmacies** | Have antihistamine stock ready by **week 13–14** in the south and middle and **week 16** in the north (birch), and by **week 20** (Malmö, Göteborg) → **22–23** (north) for grass. Roll out **south → north** with ~2–3 weeks between regions, and keep a buffer for warm springs. Göteborg also needs stock for **alder in February–March**. | Q2, Q8 |
| **Retail (home & electronics, supermarkets, e-commerce)** | Two windows: **February–March** for particle filters and purifiers in **Malmö and Göteborg** (PM2.5 = fine particles), and **April (south) → May (middle and north)** for pollen products. Instead of a fixed calendar date, **start campaigns when the season starts in each region** – and test it (see Q12: geo-experiment). | Q11, Q8 |
| **Tourism** | Market **"pollen-free spring"** by direction: the **north in April**, the **south in late May**, and **early June anywhere**. Luleå has the fewest high-pollen days over the whole spring (11%), Göteborg the most (27%). | Q9 |
| **Outdoor events** | For any outdoor event pick **late August – early September** (weeks 33–36: fewest problem days with pollen, rain and bad air). For allergy-friendly events, **early June** is also good. Avoid **mid May (week 19)** in the south and middle and **early July (week 27)**. | Q10 |

### Limitations to keep in mind
- Pollen values come from the **CAMS model** (estimates, ~11 km grid), not from measurements; alder and mugwort are weak in the model.
- Only **~6 seasons (2021–2026)**: enough for patterns by hour, weather and city, but too short to explain differences between years with confidence.
- Results are at **city level**, not neighbourhood or kommun.
- "High pollen" thresholds are an assumption based on common Nordic scales.
- **Validated against measurements (CAMM, Stockholm):** birch and grass season timing is reliable; absolute levels are about half of the measured ones, so high-pollen days are underestimated; alder is not reliable.